# Panorama da Segurança Operacional nos Aeroportos do Brasil:
## Uma Análise Baseada em Ocorrências e Exposição (2023–2024)

Uma das principais métricas utilizada para avaliar segurança operacional na aviação consiste em promover análises a partir dos dados de ocorrências aeronáuticas, como acidentes e incidentes. No entanto, análises baseadas exclusivamente em números absolutos podem levar a interpretações equivocadas, uma vez que não consideram o volume de operações ao qual o sistema está exposto.

Segundo a International Civil Aviation Organization (ICAO, 2018), a avaliação da segurança deve ser realizada com base em indicadores normalizados por exposição, como taxas de ocorrências por número de voos ou horas de voo. Essa abordagem é amplamente adotada pela indústria, sendo utilizada, por exemplo, nos relatórios da International Air Transport Association (IATA), que expressam o desempenho de segurança em termos de acidentes por milhão de voos.

Este estudo tem por objetivo verificar a exposição ao risco dos aeroportos do Brasil, não restrigindo tipos de operação, considerando o número de ocorrências comparado com o volume de movimentos em  cada unidade, ao invés de observar números absolutos. Isso nos dá um índice de risco proporcional e comparável, que torna possível a comparação da segurança operacional entre aeroportos.

### Hipóteses avaliadas
Dado o contexto de segurança operacional apresentado anteriormente, buscaremos ao longo do estudo testar as seguintes hipóteses:

**H1:** O número de ocorrências aeronáuticas não cresce proporcionalmente ao volume de operações, apresentando comportamento sublinear quando ajustado por exposição.

**H2:** Incidentes apresentam maior concentração em determinados aeroportos, enquanto acidentes e incidentes graves possuem distribuição mais uniforme.

**H3:** A utilização de métricas baseadas em valores absolutos distorce a avaliação da segurança operacional, enquanto métricas normalizadas por volume de operações permitem comparações mais adequadas entre aeroportos.

### Dados Utilizados

Neste estudo são utilizados dados disponibilizados por dois órgãos nacionais responsáveis pela gestão e fiscalização do serviço aeroportuário do Brasil: A ANAC e o CENIPA. A ANAC é a agencia reguladora responsável por auditar e fiscalizar a aviação civil no país. O CENIPA é um órgão subordinado às Força Aerea Brasileira responsável pela investigação dos incidentes e acidentes aéreos. Cada investigação gera um relatório com um conjunto de recomendações que tem como objetivo evitar que ocorrências similares aconteçam.

Toda ocorrência no serviço aéreo em território brasileiro deve ser reportada ao CENIPA. Os dados são diponibilizados seguindo a política de dados abertos do Governo Federal. Os dados são disponibilizados em formato csv e atualizados na peridicidade de conveniência à Agência.

## Agenda Resumida

### Definição do problema
- **Descrição do problema:** avaliar o risco operacional de aeroportos brasileiros com base em ocorrências do CENIPA normalizadas pela exposição operacional (Registro de Voos Ativos/ANAC).
- **Tipo de problema:** análise exploratória de dados (não supervisionado), sem variável-alvo para treinamento de modelo.
- **Premissas e hipóteses:** H1, H2 e H3 estão definidas na introdução e verificadas ao longo das análises.
- **Restrições de seleção dos dados:** foco em operações no Brasil, janela principal 2023-2024, e necessidade de integração entre bases ANAC e CENIPA por heurística de correspondência.

### Definição dos atributos principais
| Atributo | Base | Significado |
|---|---|---|
| `ocorrencia_dia` | CENIPA | Data da ocorrência |
| `ocorrencia_classificacao` | CENIPA | Classe da ocorrência (`ACIDENTE`, `INCIDENTE GRAVE`, `INCIDENTE`) |
| `ocorrencia_cidade` | CENIPA | Cidade associada à ocorrência |
| `ocorrencia_latitude` / `ocorrencia_longitude` | CENIPA | Localização geográfica da ocorrência |
| `aeronave_fase_operacao` | CENIPA | Fase do voo no momento da ocorrência |
| `aeronave_voo_origem` / `aeronave_voo_destino` | CENIPA | Aeroporto de origem/destino relacionado à aeronave |
| `Sigla ICAO Aeroporto Origem` / `Sigla ICAO Aeroporto Destino` | ANAC (VRA) | Identificador ICAO do aeroporto |
| `Descrição Aeroporto Origem` / `Descrição Aeroporto Destino` | ANAC (VRA) | Nome textual do aeroporto |
| `Situação Voo` | ANAC (VRA) | Situação do voo (ex.: `REALIZADO`) |

### Itens de análise e pré-processamento
- **Estatísticas descritivas:** apresentadas em seção dedicada com mínimo, máximo, mediana, moda, média, desvio-padrão e ausentes para atributos numéricos relevantes.
- **Valores faltantes/discrepantes:** diagnóstico por coluna e tratamento justificado estão documentados antes das análises finais.
- **Visualizações + interpretação:** cada gráfico relevante possui comentário analítico com achado, explicação e implicação para as hipóteses.
- **Pré-processamento:** conversão de tipos, filtros por ano, agregações por aeroporto, mapeamento CENIPA-ICAO e cálculo das taxas normalizadas por exposição.

# Apresentação dos dados
Os dados foram obtidos diretamente do portal de dados abertos do Governo Brasileiro e os links diretos para as fontes podem ser observados no arquivo ```fontes de coleta.rtf```

Os números das principais variáveis avaliadas ao longo desse trabalho são resultado de cruzamentos dos dados:
- do **CENIPA/FAB** — base de ocorrências aeronáuticas (acidentes, incidentes graves e incidentes) registrada pelo Centro de Investigação e Prevenção de Acidentes Aeronáuticos desde 2007.
- da **ANAC** — dados de VRA (Voos Regulares Ativos) de 2023 e 2024, com granularidade de voo individual e situação de execução. Os dados de VRA são divulgados mês a mês.

## Carregamento dos dados

| DataFrame | Shape | Fonte | Encoding | Separador |
|---|---|---|---|---|
| `df_vra_2023` | 981 206 × 8 | 12 CSVs mensais VRA 2023 (jan–dez) | UTF-8 | `;` |
| `df_vra_2024` | 987 868 × 8 | 12 CSVs mensais VRA 2024 (jan–dez) | UTF-8 | `;` |
| `df_ocorrencia` | ~13 186 × 22 | `ocorrencia.csv` (CENIPA/FAB) | latin1 | `;` |
| `df_aeronave` | ~13301, 23 | `aeronave.csv` (CENIPA/FAB) | latin1 | `;` |


As 8 colunas retidas do VRA via usecols são: ICAO da empresa, nome da empresa, assentos, ICAO de origem, descrição do aeroporto de origem, ICAO de destino, descrição do aeroporto de destino e Situação Voo

In [39]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as ptex #
from os import path
from glob import glob

'''Existem funções que foram criadas para facilitar manipulação dos dados para reduzir a redundancia de codigo
   Essas funções estao localizadas na pasta source/utils/dataframe_operations.py, que foram clonadas na celula anterior
   No colab, o carregamento dos dados e feito de forma manual, por isso é necessario executar o codigo do loob abaixo'''

for _p in (Path.cwd(), Path.cwd() / "source"):
    if (_p / "utils" / "dataframe_operations.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

# carregamento dos dados
from utils.dataframe_operations import (
    DataFrameOperations, load_datasource_urls, filter_by_year,
    build_ocorrencias_por_aeroporto,
)

'''O mvp utiliza 26 diferentes fontes de dados. A url de acesso a cada uma dessas
    fontes está armazenada no arquivo source/utils/datasources.json
    as operacoes para obter as urls estao na classe dataframe_operations.py'''
datasources = load_datasource_urls()
loader = DataFrameOperations()

df_aeronave = loader.load_dataframe(datasources["aeronave"])
df_ocorrencia = loader.load_dataframe(datasources["ocorrencia"])

In [40]:
'''carregamento dos dados do Registro de Voos ativos, anos 2023
Dados sao recebidos em formato de arquivos separados por mes. Sao concatenados em um unico dataframe
Considerando o shape desses dataframes, essa célula e a seguinte podem demorar ate 1 min para executar'''
cols=["Sigla ICAO Empresa Aérea","Empresa Aérea","Número de Assentos","Sigla ICAO Aeroporto Origem",
       "Descrição Aeroporto Origem","Sigla ICAO Aeroporto Destino","Descrição Aeroporto Destino","Chegada Real","Situação Voo"]
df_vra_2023 = loader.load_dataframe(datasources["vra_2023_01"],
datasources["vra_2023_02"],
datasources["vra_2023_03"],
datasources["vra_2023_04"],
datasources["vra_2023_05"],
datasources["vra_2023_06"],
datasources["vra_2023_07"],
datasources["vra_2023_08"],
datasources["vra_2023_09"],
datasources["vra_2023_10"],
datasources["vra_2023_11"],
datasources["vra_2023_12"],
usecols=cols,encoding="utf-8")

In [41]:
df_vra_2024 = loader.load_dataframe(datasources["vra_2024_01"],
datasources["vra_2024_02"],
datasources["vra_2024_03"],
datasources["vra_2024_04"],
datasources["vra_2024_05"],
datasources["vra_2024_06"],
datasources["vra_2024_07"],
datasources["vra_2024_08"],
datasources["vra_2024_09"],
datasources["vra_2024_10"],
datasources["vra_2024_11"],
datasources["vra_2024_12"],
usecols=cols,encoding="utf-8")

In [42]:
# sanity check de carga
print('Shapes carregados:')
print('df_ocorrencia:', df_ocorrencia.shape)
print('df_aeronave:', df_aeronave.shape)
print('df_vra_2023:', df_vra_2023.shape)
print('df_vra_2024:', df_vra_2024.shape)

Shapes carregados:
df_ocorrencia: (13185, 22)
df_aeronave: (13301, 23)
df_vra_2023: (981206, 9)
df_vra_2024: (987868, 9)


# Análise exploratória do conjunto de dados de "Ocorrências".
Na aviação, as ocorrências operacionais ou de segurança são classificadas em acidentes, incidentes graves e incidentes. Os incidentes são aqueles em que a ocorrência afeta o funcionamento e compromete a segurança operacional, porém sem danos e sem feridos. Os incidentes graves envolvem uma falha séria de segurança, mas ainda sem a ocorrência de feridos. Os acidentes são ocorrências em que acontece dano grave à aeronave e/ou feridos (CENIPA NSCA 3-13).
Vamos observar como estes dados estão dispostos por meio de algumas ferramentas de visualização. Precisamos entender a quantidade de ocorrências registradas ao longo do tempo e como elas se dividem dentro da sua classificação.

In [43]:
# contagem simples de ocorrencias por ano e contagem segmentada por tipo de ocorrencia

df_ocorrencia['ocorrencia_dia'] = pd.to_datetime(
    df_ocorrencia['ocorrencia_dia'],
    format='%d/%m/%Y',
    dayfirst=True,
    errors='coerce'
)

#evolução dos regitros das ocorrências por ano
years = df_ocorrencia['ocorrencia_dia'].dt.year
count_ocorr_ano = years.value_counts().sort_index()

ocorrencias_por_ano = ptex.bar(count_ocorr_ano)
ocorrencias_por_ano.update_layout(width=500, height=500)
ocorrencias_por_ano.show()

df_valid = df_ocorrencia[df_ocorrencia['ocorrencia_dia'].notna()].copy()
df_valid['ano'] = df_valid['ocorrencia_dia'].dt.year

# Distribuição por ano de quantas ocorrencias foram acidentes e quantas ocorrências foram incidentes
grouped_counts = (
    df_valid
    .groupby(['ano', 'ocorrencia_classificacao'])
    .size()
)

tabela = grouped_counts.reset_index(name='contagem')
tabela['ano'] = tabela['ano'].astype(str)

ocorrencias_seg_ano = ptex.bar(
    tabela,
    x='ano',
    y='contagem',
    color='ocorrencia_classificacao',
    barmode='stack',
    labels={'ano': 'Ano', 'contagem': 'Número de ocorrências', 'ocorrencia_classificacao': 'Classificação'},
    title='Ocorrências por ano segmentadas por classificação',
    width=800,
    height=500,
     color_discrete_map={
        "INCIDENTE": "#27D3F5",
        "INCIDENTE GRAVE": "#f7a831",
        "ACIDENTE": "#EF553B",
    }
)
ocorrencias_seg_ano.show()

Ao observar a evolução consolidada e estratificada do dataframe, se vê que a quantidade de ocorrências, no geral, oscilou pouco até 2022. No entanto os anos de 2023 e 2024 chamam a atenção pelo seu montate elevado. Por conta dessa discrepância com o restante do conjunto, manteremos os anos de 2023 e 2024 como foco central deste estudo.

## Estatísticas descritivas (atributo numérico Latitude e Longitude)

In [44]:
# resumo estatistico dos valores de latitude e longitude
cols_numericas_interesse = [
    c for c in [
        'ocorrencia_latitude',
        'ocorrencia_longitude',
    ]
    if c in df_ocorrencia.columns
]

if cols_numericas_interesse:
    df_num = df_ocorrencia[cols_numericas_interesse].apply(pd.to_numeric, errors='coerce')

    stats_df = pd.DataFrame({
        'atributo': df_num.columns,
        'minimo': df_num.min().values,
        'maximo': df_num.max().values,
        'mediana': df_num.median().values,
        'moda': [s.mode(dropna=True).iloc[0] if not s.mode(dropna=True).empty else pd.NA for _, s in df_num.items()],
        'media': df_num.mean().values,
        'desvio_padrao': df_num.std().values,
        'valores_ausentes': df_num.isna().sum().values,
    })

    display(stats_df.sort_values('atributo').reset_index(drop=True))
else:
    print('Nenhuma coluna numérica de interesse encontrada em df_ocorrencia.')

,atributo,minimo,maximo,mediana,moda,media,desvio_padrao,valores_ausentes
0,ocorrencia_latitude,-235.075000,95.335,-20.821944,-23.435556,-18.025891,10.208004,4262
1,ocorrencia_longitude,-81.308889,487.575,-47.269322,-46.473056,-47.085940,11.634773,4269


Pelos resultados, latitude e longitude apresentam amplitude geográfica compatível com o território brasileiro e presença de valores ausentes (ou não parseáveis), o que justifica tratamento explícito antes de análises espaciais. A moda tende a representar coordenadas recorrentes em hubs e áreas aeroportuárias com alta concentração de registros. Como próximos passos, os ausentes são diagnosticados por coluna e tratados com regra documentada para preservar a consistência das análises.

## Diagnóstico e tratamento de faltantes/inconsistências

Antes das análises espaciais e agregações, fazemos um diagnóstico objetivo de valores ausentes e não numéricos em atributos críticos. Em seguida, documentamos as regras de tratamento aplicadas e o efeito no volume de dados.

In [45]:
# diagnostico de faltantes/inconsistencias + antes vs depois
colunas_criticas = [
    c for c in [
        'ocorrencia_dia',
        'ocorrencia_pais',
        'ocorrencia_classificacao',
        'ocorrencia_cidade',
        'ocorrencia_latitude',
        'ocorrencia_longitude',
    ]
    if c in df_ocorrencia.columns
]

diag = pd.DataFrame({
    'atributo': colunas_criticas,
    'ausentes': [df_ocorrencia[c].isna().sum() for c in colunas_criticas],
    'percentual_ausente': [round(df_ocorrencia[c].isna().mean() * 100, 2) for c in colunas_criticas],
})

# inconsistência numérica em coordenadas
if 'ocorrencia_latitude' in df_ocorrencia.columns:
    lat_num = pd.to_numeric(df_ocorrencia['ocorrencia_latitude'], errors='coerce')
    diag.loc[diag['atributo'] == 'ocorrencia_latitude', 'nao_numericos'] = lat_num.isna().sum()
if 'ocorrencia_longitude' in df_ocorrencia.columns:
    lon_num = pd.to_numeric(df_ocorrencia['ocorrencia_longitude'], errors='coerce')
    diag.loc[diag['atributo'] == 'ocorrencia_longitude', 'nao_numericos'] = lon_num.isna().sum()

diag['nao_numericos'] = diag['nao_numericos'].fillna(0).astype(int)

# regras de tratamento para analise espacial
antes_total = len(df_ocorrencia)
df_espacial = df_ocorrencia.copy()

if 'ocorrencia_pais' in df_espacial.columns:
    df_espacial = df_espacial[df_espacial['ocorrencia_pais'].astype(str).str.upper() == 'BRASIL']

if 'ocorrencia_latitude' in df_espacial.columns and 'ocorrencia_longitude' in df_espacial.columns:
    df_espacial['ocorrencia_latitude'] = pd.to_numeric(df_espacial['ocorrencia_latitude'], errors='coerce')
    df_espacial['ocorrencia_longitude'] = pd.to_numeric(df_espacial['ocorrencia_longitude'], errors='coerce')
    df_espacial = df_espacial.dropna(subset=['ocorrencia_latitude', 'ocorrencia_longitude'])

depois_total = len(df_espacial)

antes_depois = pd.DataFrame([
    {'etapa': 'Antes do tratamento', 'registros': antes_total},
    {'etapa': 'Depois do filtro BR + coordenadas válidas', 'registros': depois_total},
    {'etapa': 'Registros removidos', 'registros': antes_total - depois_total},
])

print('Diagnóstico de faltantes/inconsistências (colunas críticas):')
display(diag.sort_values('percentual_ausente', ascending=False).reset_index(drop=True))
print('Resumo de impacto (antes vs depois):')
display(antes_depois)


Diagnóstico de faltantes/inconsistências (colunas críticas):


,atributo,ausentes,percentual_ausente,nao_numericos
0,ocorrencia_latitude,2760,20.93,4262
1,ocorrencia_longitude,2760,20.93,4269
2,ocorrencia_dia,0,0.00,0
3,ocorrencia_pais,0,0.00,0
4,ocorrencia_classificacao,0,0.00,0
5,ocorrencia_cidade,0,0.00,0


Resumo de impacto (antes vs depois):


,etapa,registros
0,Antes do tratamento,13185
1,Depois do filtro BR + coordenadas válidas,8903
2,Registros removidos,4282


**Regras aplicadas:** mantemos apenas ocorrências no Brasil para garantir aderência ao escopo do estudo e removemos, para análise espacial, registros sem latitude/longitude válidas. Isso evita viés visual por pontos inválidos e mantém rastreável o impacto do tratamento. As demais análises agregadas preservam o dataset original sempre que possível, reduzindo perda de informação potencialmente valiosa.

## Distribuição Espacial das Ocorrências
Considerando a variável numérica _quantidade de ocorrências_, representada nos gráficos acima, nota-se um aumento expressivo de volume nos anos de 2023 e 2024.
Por isso, seguimos mantendo esse período como alvo.

Conforme discutido na apresentação do problema, números absolutos não geram um indicativo forte para segurança operacional. Vamos observar a disposição espacial das ocorrências utilizando os atributos de latitude e longitude presentes no dataframe ```df_ocorrencia```

In [46]:
#distribuição espacial de ocorrencias por latitude e longitude, anos 2023 e 2024

import math
from utils.dataframe_operations import filter_by_year

df_ocorrencia_brasil = filter_by_year(df_espacial, 'ocorrencia_dia', [2023, 2024])

null_lat = pd.to_numeric(df_ocorrencia_brasil['ocorrencia_latitude'], errors='coerce').isna().sum()
null_lon = pd.to_numeric(df_ocorrencia_brasil['ocorrencia_longitude'], errors='coerce').isna().sum()
print(f"Nulos — latitude: {null_lat}, longitude: {null_lon}  (de {len(df_ocorrencia_brasil)} registros)")

lat_col = pd.to_numeric(df_ocorrencia_brasil["ocorrencia_latitude"], errors="coerce").dropna()
lon_col = pd.to_numeric(df_ocorrencia_brasil["ocorrencia_longitude"], errors="coerce").dropna()

lat_min, lat_max = lat_col.min(), lat_col.max()
lon_min, lon_max = lon_col.min(), lon_col.max()

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2
zoom = math.log2(360 / max(lat_max - lat_min, lon_max - lon_min)) - 1

dist_espacial_ocorrencias  = ptex.scatter_map(
    df_ocorrencia_brasil,
    lat="ocorrencia_latitude",
    lon="ocorrencia_longitude",
    hover_name="ocorrencia_cidade",
    color="ocorrencia_classificacao",
    category_orders={
        "ocorrencia_classificacao": ["ACIDENTE", "INCIDENTE GRAVE", "INCIDENTE"]
    },
    color_discrete_map={
        "INCIDENTE": "#27D3F5",
        "INCIDENTE GRAVE": "#f7a831",
        "ACIDENTE": "#EF553B",
    },
    zoom=3.3,
    center={"lat": -14, "lon": -52},
    map_style="carto-positron",
    opacity=0.6,
    title="Ocorrências Aéreas por Localização — Brasil"
)

counts = df_ocorrencia_brasil['ocorrencia_classificacao'].value_counts()
subtitle = "  ·  ".join(f"{cls}: {n}" for cls, n in counts.items())

dist_espacial_ocorrencias.update_layout(
    height=1300,
    width=1200,
    map=dict(
       # bounds=dict(west=-75, east=-34, south=-34, north=6)
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        title_text="",
        itemclick="toggle",
        itemdoubleclick="toggleothers"
    ),
    annotations=[dict(
        text=f"Total: {counts.sum()}  ·  {subtitle}",
        xref="paper", yref="paper",
        x=0.5, y=-0.03,
        showarrow=False,
        font=dict(size=13),
        xanchor="center",
        yanchor="top",
    )]
)

dist_espacial_ocorrencias.show()

Nulos — latitude: 0, longitude: 0  (de 3853 registros)


### Disposição da distribuição espacial

No mapa acima é possivel fazer toggle/untoggle de alguma classificação específica clicando sobre a legenda acima do mapa.

Infelizmente, a disposição espacial por si só não nos dá uma ideia clara de como acidentes e incidentes estão distribuídos visto que muitos pontos estão sobrepostos se considerarmos que, com frequência, as ocorrências  acontecem em coordenadas próximas (áreas de aeroportos e aeródromos). Apesar de não ser possível fazer uma afirmação categórica, percebe-se que _incidentes se sobrepõem com mais frequência que acidentes e incidentes graves_ devido à maior opacidade dos pontos azuis. Partindo dessa premissa, vamos manter em mente a possibilidade de que _acidentes tenham uma distribuição mais uniforme enquanto incidentes têm uma distribuição concentrada em certos hubs/clusters_.

Pensando em confirmar essa ideia, vamos agrupar as ocorrências por cidade, assim poderemos ter mais clareza sobre os pontos sobrepostos

In [47]:
# contagem de ocorrencias, agrupadas por cidade e separadas por tipo de ocorrencia
# esse agrupamento é consumido pela celula de subplots
from utils.dataframe_operations import filter_by_year

df_ocorrencia_recente = filter_by_year(df_ocorrencia, 'ocorrencia_dia', [2023, 2024])

df_acidentes_cidade = (
    df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_classificacao'] == 'ACIDENTE']
    .groupby('ocorrencia_cidade').size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

df_incidentes_cidade = (
    df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_classificacao'] == 'INCIDENTE']
    .groupby('ocorrencia_cidade').size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

df_incidentes_graves_cidade = (
    df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_classificacao'] == 'INCIDENTE GRAVE']
    .groupby('ocorrencia_cidade').size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

display(df_acidentes_cidade.head())
display(df_incidentes_cidade.head())
display(df_incidentes_graves_cidade.head())


,ocorrencia_cidade,contagem
39,BRASNORTE,5
228,TERESINA,4
71,CUIABÁ,4
36,BRAGANÇA PAULISTA,4
32,BOA VISTA,3


,ocorrencia_cidade,contagem
71,GUARULHOS,389
153,RIO DE JANEIRO,342
181,SÃO PAULO,268
33,CAMPINAS,208
28,BRASÍLIA,169


,ocorrencia_cidade,contagem
81,RIO DE JANEIRO,5
0,ALTAMIRA,3
42,CURITIBA,3
23,CAMPO GRANDE,3
94,SÃO DESIDÉRIO,2


In [48]:
# subplots de ocorrencias por cidade, por classificacao e por ano

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

classificacoes = ['ACIDENTE', 'INCIDENTE GRAVE', 'INCIDENTE']
anos = [2023, 2024]
cores = {'ACIDENTE': '#EF553B', 'INCIDENTE GRAVE': '#f7a831', 'INCIDENTE': '#27D3F5'}

dist_cidades_ano_classificacao = make_subplots(
    rows=len(anos), cols=len(classificacoes),
    vertical_spacing=0.15,
    horizontal_spacing=0.08,
)

for row_idx, ano in enumerate(anos, start=1):
    df_ano = df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_dia'].dt.year == ano]
    for col_idx, cls in enumerate(classificacoes, start=1):
        contagens = (
            df_ano[df_ano['ocorrencia_classificacao'] == cls]
            .groupby('ocorrencia_cidade').size()
            .reset_index(name='contagem')
            .sort_values('contagem', ascending=False)
            .reset_index(drop=True)
        )
        contagens['rank'] = range(1, len(contagens) + 1)
        dist_cidades_ano_classificacao.add_trace(
            go.Scatter(
                x=contagens['rank'],
                y=contagens['contagem'],
                mode='markers',
                marker=dict(color=cores[cls], size=6, opacity=0.7),
                text=contagens['ocorrencia_cidade'],
                hovertemplate='<b>%{text}</b><br>Rank: %{x}<br>Ocorrências: %{y}<extra></extra>',
                showlegend=False,
            ),
            row=row_idx, col=col_idx,
        )
        show_x = row_idx == len(anos)
        show_y = col_idx == 1
        dist_cidades_ano_classificacao.update_xaxes(title_text='Rank da cidade' if show_x else '', row=row_idx, col=col_idx)
        dist_cidades_ano_classificacao.update_yaxes(title_text='Ocorrências' if show_y else '', row=row_idx, col=col_idx)

legend_text = (
    '<span style="color:#EF553B">&#9632;</span> Acidente    '
    '<span style="color:#f7a831">&#9632;</span> Incidente Grave    '
    '<span style="color:#27D3F5">&#9632;</span> Incidente'
)

dist_cidades_ano_classificacao.update_layout(
    height=700, width=1100,
    title_text='Ranking ocorrências por cidade - 2023 e 2024',
    annotations=[dict(
        text=legend_text,
        xref='paper', yref='paper',
        x=0.5, y=1.06,
        showarrow=False,
        font=dict(size=13),
        xanchor='center',
    )],
    margin=dict(t=100),
)
dist_cidades_ano_classificacao.show()

De fato, confirmamos que acidentes tem uma distribuição constante e em baixo volume, enquanto incidentes tem uma distribuição concentrada em certos hubs/clusters. Fazendo hover acima dos pontos do subplot de incidentes, vemos que os pontos com maior número de incidentes são os grandes aeroportos do país. Por exemplo, a cidade do Rio de Janeiro figura no topo da contagem de ambos os anos, sendo a única com dois aeroportos de grande volume de movimentos. O Estado de São Paulo forma um hub maior com Congonhas, Guarulhos e Viracopos, no entanto, no conjunto de dados, a contagem de cada um desses aeroportos pertence a uma cidade diferente - e todos eles também figuram no topo do ranking de incidentes.

Aqui temos vista de que a hipótese H2 é verdadeira. De fato, Incidentes apresentam maior concentração em determinados aeroportos, enquanto acidentes e incidentes graves possuem distribuição mais uniforme.

## Distribuição das ocorrências por fase de voo
O dataframe ```df_aeronave``` contém outros atributos de interesse à este estudo. Ele mantem os dados do aeroporto de onde o voo decolou, o aeroporto onde estava programado o pouso e a fase de voo em que a ocorrência foi reportada. Num procedimento de avaliação de risco, é levado em conta as fases mais críticas de operação (STOLZER, 2016). Na aviação, as fases de decolagem, aproximação final e pouso são consideradas as mais críticas do voo, concentrando a maior parte das ocorrências aeronáuticas. Isso se deve à maior complexidade operacional nessas etapas, maior probabilidade de falha humana e também à menor margem de erro associada à proximidade do solo.

Estudos internacionais indicam que a maioria dos acidentes ocorre durante as fases de aproximação e pouso, seguidas pela decolagem (ICAO, 2023; Boeing, 2022). Essa evidência reforça a importância de analisar as ocorrências considerando a fase do voo, como forma de compreender melhor os fatores de risco envolvidos.

Vamos observar o agrupamento das ocorrências registradas por fase de voo para obter a confirmação do embasamento teórico citado.

In [49]:
# agrupamento de ocorrências por fase de voo
df_ocorrencia_fase_voo = df_aeronave[df_aeronave['aeronave_fase_operacao'].notna()]

df_ocorrencia_com_fase = df_ocorrencia.merge(
    df_ocorrencia_fase_voo[['codigo_ocorrencia2', 'aeronave_fase_operacao']],
    on='codigo_ocorrencia2',
    how='inner'
)
df_ocorrencia_com_fase_2023 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2023])
df_ocorrencia_com_fase_2024 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2024])

contagem_ocorrencias_por_fase_2023 = (
    df_ocorrencia_com_fase_2023
    .groupby('aeronave_fase_operacao')
    .size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

contagem_ocorrencias_por_fase_2024 = (
    df_ocorrencia_com_fase_2024
    .groupby('aeronave_fase_operacao')
    .size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

print("2023")
display(contagem_ocorrencias_por_fase_2023.head())
print("2024")
display(contagem_ocorrencias_por_fase_2024.head())

# Treemap: raiz = total (2023+2024); primeiro nível = ano; folhas = fase de operação
_df_treemap_fase = pd.concat(
    [
        contagem_ocorrencias_por_fase_2023.assign(ano="2023"),
        contagem_ocorrencias_por_fase_2024.assign(ano="2024"),
    ],
    ignore_index=True,
)
_df_treemap_fase["total_ocorrencias"] = "Ocorrências (2023–2024)"

fig_treemap_fase = ptex.treemap(
    _df_treemap_fase,
    path=["total_ocorrencias", "ano", "aeronave_fase_operacao"],
    values="contagem",
    color="contagem",
    color_continuous_scale="Blues",
    hover_data={"contagem": ":,.0f"},
)
fig_treemap_fase.update_traces(
    textinfo="label+value+percent parent",
    textfont_size=12,
    marker=dict(line=dict(width=0.5, color="white")),
)
fig_treemap_fase.update_layout(
    title="Distribuição de ocorrências por ano e fase de operação (treemap)",
    margin=dict(t=48, l=8, r=8, b=8),
    height=560,
)
fig_treemap_fase.show()


def _estrato_por_classificacao(serie):
    """Agrupa ACIDENTE + INCIDENTE GRAVE vs INCIDENTE; demais valores em 'Outras classificações'."""

    def _one(x):
        if pd.isna(x):
            return "(sem classificação)"
        x = str(x).strip()
        if x in ("ACIDENTE", "INCIDENTE GRAVE"):
            return "Acidentes e incidentes graves"
        if x == "INCIDENTE":
            return "Incidentes"
        return "Outras classificações"

    return serie.map(_one)


_parts_treemap_estrato = []
for _df_ano, _ano in (
    (df_ocorrencia_com_fase_2023, "2023"),
    (df_ocorrencia_com_fase_2024, "2024"),
):
    _t = _df_ano.assign(estrato=_estrato_por_classificacao(_df_ano["ocorrencia_classificacao"]))
    _parts_treemap_estrato.append(
        _t.groupby(["aeronave_fase_operacao", "estrato"], dropna=False)
        .size()
        .reset_index(name="contagem")
        .assign(ano=_ano)
    )

_df_treemap_fase_estrato = pd.concat(_parts_treemap_estrato, ignore_index=True)
_df_treemap_fase_estrato["total_ocorrencias"] = "Ocorrências (2023–2024)"

fig_treemap_fase_estrato = ptex.treemap(
    _df_treemap_fase_estrato,
    path=["total_ocorrencias", "ano", "aeronave_fase_operacao", "estrato"],
    values="contagem",
    color="contagem",
    color_continuous_scale="Blues",
    hover_data={"contagem": ":,.0f"},
)
fig_treemap_fase_estrato.update_traces(
    textinfo="label+value+percent parent",
    textfont_size=12,
    marker=dict(line=dict(width=0.5, color="white")),
)
fig_treemap_fase_estrato.update_layout(
    title="Ocorrências por ano, fase de operação e gravidade (acidentes/graves vs incidentes) — treemap",
    margin=dict(t=48, l=8, r=8, b=8),
    height=560,
)
fig_treemap_fase_estrato.show()

2023


,aeronave_fase_operacao,contagem
18,POUSO,451
8,DECOLAGEM,353
7,CRUZEIRO,151
1,APROXIMAÇÃO FINAL,143
22,TÁXI,56


2024


,aeronave_fase_operacao,contagem
17,POUSO,618
5,CRUZEIRO,551
6,DECOLAGEM,452
20,REVISÃO DE PISTA,265
0,APROXIMAÇÃO FINAL,243


A distribuição por fase reforça o comportamento esperado na literatura: aproximação, pouso e decolagem concentram boa parte dos registros. Isso sugere que o risco operacional observado está associado a etapas de maior carga de trabalho e menor margem de recuperação. 

Junto com essa confirmação, emergiu outra característica interessante a respeito da distribuição pode fases de voo. A fase de cruzeiro figura no top 3 eu número de ocorrências nos dois anos. Essa proporção se justifica porque é a fase mais longa na janela temporal de um voo. Relatórios de segurança operacional reportam que em um voo comercial com 1h30min de duração, 57% do tempo a aeronave permanece em fase de cruzeiro (BOEING,2025), portanto, com mais tempo de exposição se acumulam mais ocorrências. Contudo, isso não significa que essa é uma fase menos segura que operações próximas ao solo. A fase de cruzeiro apresenta menor taxa de ocorrências críticas devido à redução da complexidade operacional, menor incidência de falha humana (maior uso de automação) e maior margem de recuperação proporcionada pela altitude. Portanto, para esta porção da avaliação, nos mantemos alinhados com a literatura sobre o assunto. 

Seguiremos considerando o mesmo conjunto de operações críticas e os dados gerados nesse tratamento serão usados na próxima fase. Para mapear a qual aeroporto a ocorrência está vinculada, usaremos uma heuristica baseada nas fases de voo.

## Construção da base de ocorrências por aeroporto (2023 e 2024)

Nesta etapa, o objetivo é preparar uma visão agregada de ocorrências por aeroporto para cada ano analisado. Para isso, a célula abaixo executa três passos:

1. filtra registros com fase de voo preenchida (`aeronave_fase_operacao`);
2. faz o merge com `df_ocorrencia` via `codigo_ocorrencia2` para trazer a fase para cada ocorrência;
3. aplica recorte anual (2023/2024) e gera a contagem por aeroporto e classificação com `build_ocorrencias_por_aeroporto`.

O resultado dessa célula é a criação dos dataframes `contagem_ocorrencias_aeroporto_2023` e `contagem_ocorrencias_aeroporto_2024`, que serão usados no cruzamento posterior com os dados de movimentação da ANAC.

In [50]:
df_ocorrencia_fase_voo = df_aeronave[df_aeronave['aeronave_fase_operacao'].notna()]

df_ocorrencia_com_fase = df_ocorrencia.merge(
    df_ocorrencia_fase_voo[['codigo_ocorrencia2', 'aeronave_fase_operacao']],
    on='codigo_ocorrencia2',
    how='inner'
)
df_ocorrencia_com_fase_2023 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2023])
df_ocorrencia_com_fase_2024 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2024])

print("2023:", df_ocorrencia_com_fase_2023.shape)
print("2024:", df_ocorrencia_com_fase_2024.shape)

contagem_ocorrencias_aeroporto_2023 = build_ocorrencias_por_aeroporto(df_ocorrencia_com_fase_2023, df_aeronave)
contagem_ocorrencias_aeroporto_2024 = build_ocorrencias_por_aeroporto(df_ocorrencia_com_fase_2024, df_aeronave)

# REMOVA OS MARCADORES PARA EXPORTAR OS DATAFRAMES
'''from utils.paths import PROCESSED_DIR
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
_out23 = PROCESSED_DIR / "contagem_ocorrencias_aeroporto_2023.csv"
_out24 = PROCESSED_DIR / "contagem_ocorrencias_aeroporto_2024.csv"
contagem_ocorrencias_aeroporto_2023.to_csv(_out23, sep=";", index=False, encoding="utf-8")
contagem_ocorrencias_aeroporto_2024.to_csv(_out24, sep=";", index=False, encoding="utf-8")
print("Exportado:", _out23)
print("Exportado:", _out24)'''

print("2023")
display(contagem_ocorrencias_aeroporto_2023)
print("2024")
display(contagem_ocorrencias_aeroporto_2024)


2023: (1396, 23)
2024: (2701, 23)
2023


,aeroporto,ocorrencia_classificacao,contagem
0,VIRACOPOS,INCIDENTE,73
1,GOVERNADOR ANDRÉ FRANCO MONTORO,INCIDENTE,71
2,CONGONHAS,INCIDENTE,64
3,SALGADO FILHO,INCIDENTE,61
4,PRESIDENTE JUSCELINO KUBITSCHEK,INCIDENTE,47
...,...,...,...
202,FAZENDA FORTALEZA DO GUAPORÉ,INCIDENTE,1
203,FAZENDA IROHY,ACIDENTE,1
204,FAZENDA NOSSA SENHORA DE FÁTIMA,ACIDENTE,1
205,FAZENDA TRADIÇÃO,INCIDENTE,1


2024


,aeroporto,ocorrencia_classificacao,contagem
0,GOVERNADOR ANDRÉ FRANCO MONTORO,INCIDENTE,145
1,ANTONIO CARLOS JOBIM / GALEÃO,INCIDENTE,93
2,CONGONHAS,INCIDENTE,72
3,GUARARAPES - GILBERTO FREYRE,INCIDENTE,72
4,PRESIDENTE JUSCELINO KUBITSCHEK,INCIDENTE,65
...,...,...,...
222,JATAÍ,INCIDENTE,1
223,BAURU,ACIDENTE,1
224,JOÃO MONTEIRO,INCIDENTE,1
225,Comandante Rolim Adolfo Amaro,INCIDENTE GRAVE,1


As saídas exibidas nesta etapa servem para validar se a base anual de ocorrências por aeroporto foi construída corretamente (quantidade de registros por ano e estrutura final dos dataframes). Seguimos para a etapa de exposição operacional, onde as contagens de ocorrências serão combinadas com a movimentação de voos para cálculo das taxas de risco.

## Análise dos dados do Registro de Voos Ativos (VRA/ANAC)

Conforme brevemente abordado na sessão anterior, existem parâmetros a se considerar durante a avaliação do risco de uma operação crítica. Nesta abordagem, a avaliação da segurança operacional deve considerar não apenas o número de ocorrências, mas também o nível de exposição do sistema às operações aeronáuticas.

De acordo com a International Civil Aviation Organization (ICAO, 2018), indicadores de segurança devem ser normalizados por medidas de atividade, como número de voos ou horas de voo, de forma a refletir adequadamente o risco operacional. De maneira complementar, a Federal Aviation Administration (FAA, 2016) destaca que o risco é função não apenas da probabilidade e severidade de eventos, mas também da exposição ao sistema.

Até aqui tratamos com informações quantitativas. A partir de agora **mantenha o cinto de segurança afivelado até que o sinal luminoso seja apagado**. Vamos dar mais um passo para a verificação das hipóteses H1 e H3.

### Tratamento dos dados: Contagem de movimentos por aeroporto
Nos dataframes ```df_vra_2023```  e  ```df_vra_2024``` estão registrados todos os voos programados que tenham como origem e/ou destino no Brasil. Para fazer a contagem de movimentos, foram consideradas as linhas onde a Situação do Voo = ```REALIZADO```. Para os aeroportos que recebem e realizam voos internacionais, contabilizamos apenas o movimento ocorrido aeroporto do Brasil para manter os dados concisos de acordo com o objetivo do estudo.

O número de movimentos por aeroporto será o nosso fator exposição.





In [51]:
#contagem de movimentos VRA por aeroporto indexada por sigla ICAO (para cruzamento com CENIPA)
# Sigla ICAO é um identificador único para cada aeroporto no mundo, registrado na International Civil Aviation Organization (ICAO)

def _group_VRA_BR_icao(df):
    from collections import defaultdict
    group_by_icao = defaultdict(lambda: {"pousos": 0, "decolagens": 0})

    ICAO_ORIGEM  = "Sigla ICAO Aeroporto Origem"
    ICAO_DESTINO = "Sigla ICAO Aeroporto Destino"
    DESC_ORIGEM  = "Descrição Aeroporto Origem"
    DESC_DESTINO = "Descrição Aeroporto Destino"
    SITUACAO_VOO = "Situação Voo"

    cols = list(df.columns)
    i_sit       = cols.index(SITUACAO_VOO)
    i_icao_orig = cols.index(ICAO_ORIGEM)
    i_icao_dest = cols.index(ICAO_DESTINO)
    i_desc_orig = cols.index(DESC_ORIGEM)
    i_desc_dest = cols.index(DESC_DESTINO)

    for row in df.itertuples(index=False, name=None):
        if row[i_sit] != "REALIZADO":
            continue

        desc_orig = row[i_desc_orig]
        desc_dest = row[i_desc_dest]
        icao_orig = row[i_icao_orig]
        icao_dest = row[i_icao_dest]

        if pd.notna(desc_orig) and "brasil" in str(desc_orig).lower() and pd.notna(icao_orig):
            group_by_icao[str(icao_orig).strip()]["decolagens"] += 1
        if pd.notna(desc_dest) and "brasil" in str(desc_dest).lower() and pd.notna(icao_dest):
            group_by_icao[str(icao_dest).strip()]["pousos"] += 1

    out = pd.DataFrame.from_dict(group_by_icao, orient="index")
    out["movimentacao_total"] = out["pousos"] + out["decolagens"]
    return out.sort_values("movimentacao_total", ascending=False)

contagem_movimentos_br_icao_2023 = _group_VRA_BR_icao(df_vra_2023)
contagem_movimentos_br_icao_2024 = _group_VRA_BR_icao(df_vra_2024)

print("2023:", contagem_movimentos_br_icao_2023.shape)
display(contagem_movimentos_br_icao_2023.head(10))
print("2024:", contagem_movimentos_br_icao_2024.shape)
display(contagem_movimentos_br_icao_2024.head(10))

# Histogramas — contagem de aeroportos por faixa de movimentacao_total agregada
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_fig_hist_simple = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("2023", "2024"),
)
for _col, (_df_mov, _ano) in enumerate(
    (
        (contagem_movimentos_br_icao_2023, "2023"),
        (contagem_movimentos_br_icao_2024, "2024"),
    ),
    start=1,
):
    _x = _df_mov["movimentacao_total"].astype(float)
    _fig_hist_simple.add_trace(
        go.Histogram(
            x=_x,
            nbinsx=35,
            name=_ano,
            marker_color="#636EFA",
            opacity=0.85,
            showlegend=False,
        ),
        row=1,
        col=_col,
    )
_fig_hist_simple.update_xaxes(title_text="movimentacao_total (pousos + decolagens)", row=1, col=1)
_fig_hist_simple.update_xaxes(title_text="", row=1, col=2)
_fig_hist_simple.update_yaxes(title_text="Frequência (nº de aeroportos no intervalo)", row=1, col=1)
_fig_hist_simple.update_yaxes(title_text="", row=1, col=2)
_fig_hist_simple.update_layout(
    height=400,
    title_text="Histograma — movimentação agregada por aeroporto ICAO (Brasil)",
    title_x=0.5,
)
_fig_hist_simple.show()

2023: (191, 3)


,pousos,decolagens,movimentacao_total
SBGR,131339,131391,262730
SBSP,92987,92991,185978
SBKP,62075,62083,124158
SBBR,55304,55307,110611
SBRJ,53344,53365,106709
SBCF,48534,48535,97069
SBRF,38159,38091,76250
SBPA,30534,30525,61059
SBSV,27606,27596,55202
SBCT,26446,26656,53102


2024: (191, 3)


,pousos,decolagens,movimentacao_total
SBGR,136722,136735,273457
SBSP,94257,94262,188519
SBKP,59851,59916,119767
SBCF,55894,55899,111793
SBBR,54516,54509,109025
SBGL,48762,48785,97547
SBRF,41143,41092,82235
SBRJ,28583,28620,57203
SBSV,27653,27645,55298
SBCT,25953,26162,52115


A tabela e a figura confirmam a forte concentração de movimentos em poucos hubs nacionais, com cauda longa de aeroportos de baixo volume. Esse padrão assimétrico reforça por que comparar segurança por contagens absolutas tende a distorcer conclusões. Portanto, os próximos cálculos usam taxas normalizadas por movimentação para permitir comparação justa entre aeroportos de portes distintos.

In [52]:
# evolução mensal de ocorrências normalizadas por movimentos (2023-2024)
# taxa mensal = total de ocorrências no mês / total de movimentos no mês

import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_oc = df_ocorrencia.copy()
df_oc['ocorrencia_dia'] = pd.to_datetime(
    df_oc['ocorrencia_dia'], format='%d/%m/%Y', dayfirst=True, errors='coerce'
)
df_oc = df_oc[df_oc['ocorrencia_dia'].notna()]
df_oc = df_oc[df_oc['ocorrencia_dia'].dt.year.isin([2023, 2024])]
df_oc['ano_mes'] = df_oc['ocorrencia_dia'].dt.to_period('M').dt.to_timestamp()

ocorrencias_mensais = (
    df_oc.groupby('ano_mes')
    .size()
    .reset_index(name='total_ocorrencias')
    .sort_values('ano_mes')
)

# total de movimentos por mês a partir do VRA (2023 e 2024)
df_mov = pd.concat([df_vra_2023, df_vra_2024], ignore_index=True).copy()
col_data_candidates = [
    'Chegada Real',
    'Partida Real',
    'Partida Prevista',
    'Chegada Prevista',
    'Data Voo',
    'Data do Voo',
    'Data',
]
col_data_mov = next((c for c in col_data_candidates if c in df_mov.columns), None)
if col_data_mov is None:
    raise ValueError(
        f'Nenhuma coluna de data encontrada no VRA. Colunas disponíveis: {list(df_mov.columns)}'
    )

serie_data = pd.to_datetime(df_mov[col_data_mov], dayfirst=True, errors='coerce')
if serie_data.isna().all():
    serie_data = pd.to_datetime(df_mov[col_data_mov], errors='coerce')

df_mov['data_movimento'] = serie_data

if 'Situação Voo' in df_mov.columns:
    df_mov = df_mov[df_mov['Situação Voo'] == 'REALIZADO']

df_mov = df_mov[df_mov['data_movimento'].notna()]
df_mov['ano_mes'] = df_mov['data_movimento'].dt.to_period('M').dt.to_timestamp()

movimentos_mensais = (
    df_mov.groupby('ano_mes')
    .size()
    .reset_index(name='total_movimentos')
    .sort_values('ano_mes')
)

# taxa correta: ocorrências / movimentos
serie_plot = ocorrencias_mensais.merge(movimentos_mensais, on='ano_mes', how='inner')
serie_plot['taxa_ocorrencias'] = serie_plot['total_ocorrencias'] / serie_plot['total_movimentos']

_oc_por_mes_cls = df_oc.groupby(['ano_mes', 'ocorrencia_classificacao']).size().unstack(fill_value=0)
for _c in ('ACIDENTE', 'INCIDENTE GRAVE', 'INCIDENTE'):
    if _c not in _oc_por_mes_cls.columns:
        _oc_por_mes_cls[_c] = 0
_oc_por_mes_cls = _oc_por_mes_cls[['ACIDENTE', 'INCIDENTE GRAVE', 'INCIDENTE']]
serie_plot = serie_plot.merge(_oc_por_mes_cls.reset_index(), on='ano_mes', how='inner')

mov_mes = movimentos_mensais.copy()
mov_mes['ano'] = mov_mes['ano_mes'].dt.year
mov_mes['mes_lab'] = mov_mes['ano_mes'].dt.strftime('%b').str.title()

fig_mov_ano = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('2023', '2024'),
    horizontal_spacing=0.06,
)

for col, ano in enumerate([2023, 2024], start=1):
    sub = mov_mes[mov_mes['ano'] == ano].sort_values('ano_mes')
    fig_mov_ano.add_trace(
        go.Bar(
            x=sub['mes_lab'],
            y=sub['total_movimentos'],
            marker_color='#0d214f',
            opacity=0.85,
            hovertemplate='<b>%{customdata}</b><br>Movimentos: %{y:,.0f}<extra></extra>',
            customdata=sub['ano_mes'].dt.strftime('%b/%Y'),
            showlegend=False,
        ),
        row=1,
        col=col,
    )

fig_mov_ano.update_layout(
    title='Movimentos mensais (VRA), por ano',
    height=500,
    barmode='group',
)
fig_mov_ano.update_yaxes(nticks=5, title_text='Total de movimentos', row=1, col=1)
fig_mov_ano.update_yaxes(nticks=5, title_text='', row=1, col=2)
fig_mov_ano.show()

# linhas: taxa overall + taxas por classificação (legenda clicável para exibir/ocultar cada série)
_cls_colors = {
    'ACIDENTE': '#EF553B',
    'INCIDENTE GRAVE': '#f7a831',
    'INCIDENTE': '#27D3F5',
}

fig_taxa_tempo = go.Figure()
fig_taxa_tempo.add_trace(
    go.Scatter(
        x=serie_plot['ano_mes'],
        y=serie_plot['taxa_ocorrencias'],
        mode='lines+markers',
        name='Taxa overall (ocorrências/movimentos)',
        customdata=np.stack(
            [
                serie_plot['total_ocorrencias'].to_numpy(),
                serie_plot['total_movimentos'].to_numpy(),
            ],
            axis=-1,
        ),
        line=dict(color='#0d214f', width=3.0, dash='solid'),
        marker=dict(size=6, color='#0d214f'),
        hovertemplate=(
            '<b>%{x|%b/%Y}</b><br>'
            'tx overall: %{y:.2%}<br>'
            'ocorrencias/movimentos: %{customdata[0]:.0f}/%{customdata[1]:.0f}'
            '<extra></extra>'
        ),
    )
)
for _nome, _col in (
    ('ACIDENTE', 'ACIDENTE'),
    ('INCIDENTE GRAVE', 'INCIDENTE GRAVE'),
    ('INCIDENTE', 'INCIDENTE'),
):
    _taxa_cls = serie_plot[_col] / serie_plot['total_movimentos']
    fig_taxa_tempo.add_trace(
        go.Scatter(
            x=serie_plot['ano_mes'],
            y=_taxa_cls,
            mode='lines+markers',
            name=_nome,
            customdata=np.stack(
                [serie_plot[_col].to_numpy(), serie_plot['total_movimentos'].to_numpy()],
                axis=-1,
            ),
            line=dict(color=_cls_colors[_col], width=2.0, dash='solid'),
            marker=dict(size=5, color=_cls_colors[_col]),
            opacity=0.38,
            hovertemplate=(
                '<b>%{x|%b/%Y}</b><br>'
                f'{_nome}: %{{y:.2%}}<br>'
                'ocorrências/movimentos: %{customdata[0]:.0f}/%{customdata[1]:.0f}'
                '<extra></extra>'
            ),
        )
    )

fig_taxa_tempo.update_layout(
    title='Evolução mensal da taxa de ocorrências (ocorrências/movimentos)',
    xaxis_title='Mês',
    yaxis=dict(title='Taxa de ocorrências no mês', tickformat='.2%'),
    legend=dict(
        orientation='h',
        y=1.14,
        x=0,
        traceorder='normal',
        itemclick='toggle',
        itemdoubleclick='toggleothers',
    ),
    height=480,
)
fig_taxa_tempo.show()

A evolução mensal da taxa overall (ocorrências/movimentos) mostra variação moderada ao longo de 2023 e 2024, sem uma tendência clara de alta ou de queda no período completo, o que sugere um comportamento sensível a sazonalidade operacional e a choques pontuais de ocorrência. Quando observada em conjunto com o histograma de movimentos, nota-se que meses com maior volume de operações não necessariamente apresentam maior taxa relativa, reforçando a importância de normalizar por exposição para evitar interpretações baseadas apenas em contagens absolutas. Como ponto de atenção, oscilações mais abruptas em meses específicos podem refletir baixa robustez amostral (menos ocorrências no numerador) ou mudanças de registro/reporte.

Em tempo, as linhas temporais para a evolução do total de acidentes e incidentes graves cofirmam a opção de condensar esses dois tipos de ocorrência numa mesma variavel, pois o comportamento das duas é semelhante

## Cálculo do índice de exposição ao risco

Neste ponto, portanto, temos o risco e a exposição. Com total de movimentos e o total de ocorrências, agrupado pelo aeroporto e separados pela classificação da ocorrência, podemos calcular os índices de risco normalizado por número de voos ao invés de considerar números absolutos.

### Tratamento de dados: _Fuzzy Matching_ e join dos dados ANAC e CENIPA

Esta célula prepara a chave de integração entre CENIPA (nome do aeroporto) e ANAC/VRA (código ICAO), para viabilizar o join na etapa de cálculo de taxa.

Fluxo executado na célula:

1. **Normalização textual**: remove acentos, padroniza para maiúsculas e limpa sufixos como `- Brasil`.
2. **Lookup ANAC**: combina `df_vra_2023` e `df_vra_2024` e cria dicionários `ICAO -> descrição normalizada` e `ICAO -> nome de exibição`.
3. **Match CENIPA -> ICAO**: para cada nome em `contagem_ocorrencias_aeroporto_2023/2024`, aplica correspondência por substring (incluindo nomes compostos por `/`) e seleciona o candidato com maior score.
4. **Saída para join**: grava a coluna `icao` nos dois dataframes anuais de ocorrências e imprime o total mapeado por ano.
5. **Controle de qualidade**: lista os aeroportos não mapeados (no código atual, impressão explícita para 2023) para revisão manual.

Com isso, as tabelas de ocorrências passam a ter o identificador ICAO necessário para o cruzamento com os dados de movimentação da ANAC.

In [53]:
#mapeamento de nomes de aeroporto CENIPA → ICAO via descrição VRA
#CENIPA usa nomes completos (ex: "VIRACOPOS"), VRA usa codigos ICAO (ex: "SBKP")
#a ponte é a coluna de descrição VRA, que contem o nome do aeroporto embutido

import unicodedata

def _normalize_str(s):
    """Remove acentos e converte para maiusculas."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(s).upper())
        if unicodedata.category(c) != 'Mn'
    )

def _build_icao_lookup(df_vra):
    """Constrói dict ICAO → descrição normalizada, filtrado para aeroportos brasileiros."""
    icao_desc = {}
    for col_icao, col_desc in [
        ('Sigla ICAO Aeroporto Origem',  'Descrição Aeroporto Origem'),
        ('Sigla ICAO Aeroporto Destino', 'Descrição Aeroporto Destino'),
    ]:
        pairs = df_vra[[col_icao, col_desc]].dropna().drop_duplicates()
        mask = pairs[col_desc].str.contains('brasil', case=False, na=False)
        for icao, desc in zip(pairs.loc[mask, col_icao], pairs.loc[mask, col_desc]):
            icao = str(icao).strip()
            if len(icao) == 4 and icao not in icao_desc:
                norm = _normalize_str(desc)
                norm = norm.replace('- BRASIL', '').replace('-BRASIL', '').strip()
                icao_desc[icao] = norm
    return icao_desc

def _match_cenipa_to_icao(cenipa_name, icao_desc):
    """Encontra o ICAO para um nome de aeroporto CENIPA por correspondência de substring.

    Separa nomes compostos por '/' (ex: "ANTONIO CARLOS JOBIM / GALEÃO") e
    pontua cada ICAO candidato pelo total de caracteres correspondidos.
    """
    parts = [p.strip() for p in _normalize_str(cenipa_name).split('/')]
    best_icao, best_score = None, 0
    for icao, norm_desc in icao_desc.items():
        score = sum(len(p) for p in parts if len(p) >= 4 and p in norm_desc)
        if score > best_score:
            best_score, best_icao = score, icao
    return best_icao

def _build_icao_to_nome(df_vra):
    """Constrói dict ICAO → nome de exibição limpo (descrição VRA sem o sufixo '- Brasil')."""
    icao_nome = {}
    for col_icao, col_desc in [
        ('Sigla ICAO Aeroporto Origem',  'Descrição Aeroporto Origem'),
        ('Sigla ICAO Aeroporto Destino', 'Descrição Aeroporto Destino'),
    ]:
        pairs = df_vra[[col_icao, col_desc]].dropna().drop_duplicates()
        mask = pairs[col_desc].str.contains('brasil', case=False, na=False)
        for icao, desc in zip(pairs.loc[mask, col_icao], pairs.loc[mask, col_desc]):
            icao = str(icao).strip()
            if len(icao) == 4 and icao not in icao_nome:
                nome = str(desc).strip()
                for suffix in [' - Brasil', '- Brasil', ' – Brasil', '– Brasil', '-Brasil']:
                    if nome.lower().endswith(suffix.lower()):
                        nome = nome[:-len(suffix)].strip()
                        break
                icao_nome[icao] = nome
    return icao_nome

# Combina VRA 2023 e 2024 para maximizar cobertura do mapeamento
_vra_combined = pd.concat([df_vra_2023, df_vra_2024], ignore_index=True)
_icao_lookup   = _build_icao_lookup(_vra_combined)
_icao_to_nome  = _build_icao_to_nome(_vra_combined)

# Aplica o mapeamento: adiciona coluna 'icao' nas tabelas de ocorrências CENIPA
for df_ocorr, label in [
    (contagem_ocorrencias_aeroporto_2023, '2023'),
    (contagem_ocorrencias_aeroporto_2024, '2024'),
]:
    df_ocorr['icao'] = df_ocorr['aeroporto'].apply(
        lambda n: _match_cenipa_to_icao(n, _icao_lookup)
    )
    mapped = df_ocorr['icao'].notna().sum()
    print(f"{label}: {mapped}/{len(df_ocorr)} registros mapeados para ICAO")

unmapped = contagem_ocorrencias_aeroporto_2023.loc[
    contagem_ocorrencias_aeroporto_2023['icao'].isna(), 'aeroporto'
].unique()
if len(unmapped):
    print(f"\nNão mapeados (2023): {unmapped}")

2023: 111/207 registros mapeados para ICAO
2024: 114/227 registros mapeados para ICAO

Não mapeados (2023): <StringArray>
[                   'TENENTE-CORONEL AVIADOR CÉSAR BOMBONATO',
                                   'AERÓDROMO NÃO CADASTRADO',
                                        'CAMPO DE MARTE - SP',
                                          'FORA DE AERODROMO',
                                'AEROCLUBE DE SANTA CATARINA',
                              'Comandante Rolim Adolfo Amaro',
                        'ÁREA DE POUSO PARA USO AEROAGRÍCOLA',
 'ESTADUAL DE CAMPOS DOS AMARAIS - PREFEITO FRANCISCO AMARAL',
                                           'NÃO IDENTIFICADO',
                                     'JOÃO SIMÕES LOPES NETO',
                                              'JOÃO MONTEIRO',
                                             'POUSO DA ÁGUIA',
                                                    'RECREIO',
                          'CAVU - CLUBE DE AVIAÇÃO ULTRALEV

Aqui temos mais um indicador da instabilidade da amostra. Apenas 52% do conjunto total de ocorrências pôde ser mapeado para vinculação ao aeroporto específico. Isso acontece porque, além de termos fases de voo nao explicitamentes vinculadas a algum aeroporto (por exemplo, cruzeiro), outros não alcançaram uma pontuação mínima na heurística de matching.

### Cálculo dos índices de risco

Com a coluna `icao` já criada nas ocorrências (CENIPA) e a movimentação total por `icao` já consolidada no VRA (ANAC), a próxima célula realiza o cruzamento entre essas duas bases para calcular os indicadores de risco por aeroporto.

Definições usadas no cálculo:

- `total_grave = ACIDENTE + INCIDENTE GRAVE`
- `taxa_grave = total_grave / movimentacao_total`
- `taxa_incidente = INCIDENTE / movimentacao_total`
- `total_ocorrencias = total_grave + INCIDENTE`
- `taxa_total = total_ocorrencias / movimentacao_total`

Em termos gerais, a taxa de risco segue a forma $$ R = \frac{N}{M} $$ onde `N` é o número de ocorrências e `M` é a exposição operacional (pousos + decolagens). Esse formato é apropriado para comparação entre aeroportos de portes distintos, pois corrige o efeito de escala dos volumes absolutos.

In [54]:
#cruzamento CENIPA x VRA: calculo de taxa de ocorrencia por aeroporto
#join feito via coluna 'icao' (resolvida pelo mapeamento de nomes na celula anterior)

def _build_taxa_aeroporto(contagem_ocorrencias, contagem_movimentos):
    pivot = (
        contagem_ocorrencias.dropna(subset=['icao'])
        .pivot_table(index='icao', columns='ocorrencia_classificacao', values='contagem', aggfunc='sum')
        .fillna(0)
        .reset_index()
    )
    pivot.columns.name = None
    pivot = pivot.rename(columns={'icao': 'aeroporto'})

    for col in ['ACIDENTE', 'INCIDENTE GRAVE', 'INCIDENTE']:
        if col not in pivot.columns:
            pivot[col] = 0

    pivot['total_grave']     = pivot['ACIDENTE'] + pivot['INCIDENTE GRAVE']
    pivot['total_incidente'] = pivot['INCIDENTE']

    df = pivot.merge(
        contagem_movimentos[['movimentacao_total']],
        left_on='aeroporto',
        right_index=True,
        how='inner'
    )

    mov = df['movimentacao_total'].astype('float64')
    if (mov <= 0).any():
        print(
            'Aviso: movimentacao_total <= 0 em',
            int((mov <= 0).sum()),
            'linha(s); taxas como NaN (divisão por zero evitada).',
        )
    safe_mov = mov.where(mov > 0)
    df['taxa_grave']        = df['total_grave']       / safe_mov
    df['taxa_incidente']    = df['total_incidente']   / safe_mov
    df['total_ocorrencias'] = df['total_grave']       + df['total_incidente']
    df['taxa_total']        = df['total_ocorrencias'] / safe_mov
    df['nome_aeroporto']    = df['aeroporto'].map(_icao_to_nome).fillna(df['aeroporto'])

    return df.reset_index(drop=True)

df_taxa_aeroporto_2023 = _build_taxa_aeroporto(contagem_ocorrencias_aeroporto_2023, contagem_movimentos_br_icao_2023)
df_taxa_aeroporto_2024 = _build_taxa_aeroporto(contagem_ocorrencias_aeroporto_2024, contagem_movimentos_br_icao_2024)

print("2023 — aeroportos cruzados:", len(df_taxa_aeroporto_2023))
display(df_taxa_aeroporto_2023.sort_values('taxa_total', ascending=False).head(10))
print("2024 — aeroportos cruzados:", len(df_taxa_aeroporto_2024))
display(df_taxa_aeroporto_2024.sort_values('taxa_total', ascending=False).head(10))

2023 — aeroportos cruzados: 86


,aeroporto,ACIDENTE,INCIDENTE,INCIDENTE GRAVE,total_grave,total_incidente,movimentacao_total,taxa_grave,taxa_incidente,total_ocorrencias,taxa_total,nome_aeroporto
6,SBBI,0.0,4.0,1.0,1.0,4.0,1,1.000000,4.000000,5.0,5.000000,BACACHERI - CURITIBA - PR
7,SBBP,1.0,3.0,0.0,1.0,3.0,4,0.250000,0.750000,4.0,1.000000,ESTADUAL ARTHUR SIQUEIRA - BRAGANÇA PAULISTA - SP
33,SBJF,0.0,1.0,0.0,0.0,1.0,2,0.000000,0.500000,1.0,0.500000,FRANCISCO DE ASSIS - JUIZ DE FORA - MG
85,SWTS,0.0,0.0,1.0,1.0,0.0,4,0.250000,0.000000,1.0,0.250000,TANGARÁ DA SERRA - TANGARÁ DA SERRA - MT
72,SDCO,0.0,2.0,1.0,1.0,2.0,22,0.045455,0.090909,3.0,0.136364,SOROCABA - SOROCABA - SP
76,SSBL,1.0,0.0,0.0,1.0,0.0,12,0.083333,0.000000,1.0,0.083333,BLUMENAU - BLUMENAU - SC
5,SBBH,0.0,9.0,0.0,0.0,9.0,182,0.000000,0.049451,9.0,0.049451,PAMPULHA - CARLOS DRUMMOND DE ANDRADE - BELO H...
81,SWBC,2.0,0.0,0.0,2.0,0.0,126,0.015873,0.000000,2.0,0.015873,BARCELOS - BARCELOS - AM
61,SBSJ,0.0,1.0,1.0,1.0,1.0,240,0.004167,0.004167,2.0,0.008333,PROFESSOR URBANO ERNESTO STUMPF - SÃO JOSÉ DOS...
77,SSCN,1.0,0.0,1.0,2.0,0.0,251,0.007968,0.000000,2.0,0.007968,CANELA - CANELA - RS


2024 — aeroportos cruzados: 90


,aeroporto,ACIDENTE,INCIDENTE,INCIDENTE GRAVE,total_grave,total_incidente,movimentacao_total,taxa_grave,taxa_incidente,total_ocorrencias,taxa_total,nome_aeroporto
6,SBBP,1.0,2.0,0.0,1.0,2.0,2,0.500000,1.000000,3.0,1.500000,ESTADUAL ARTHUR SIQUEIRA - BRAGANÇA PAULISTA - SP
75,SDAG,2.0,0.0,0.0,2.0,0.0,4,0.500000,0.000000,2.0,0.500000,ANGRA DOS REIS - ANGRA DOS REIS - RJ
77,SDJV,0.0,0.0,1.0,1.0,0.0,6,0.166667,0.000000,1.0,0.166667,MUNICIPAL DE SÃO JOÃO DA BOA VISTA - SÃO JOÃO ...
76,SDCO,0.0,4.0,0.0,0.0,4.0,35,0.000000,0.114286,4.0,0.114286,SOROCABA - SOROCABA - SP
81,SSBN,0.0,1.0,0.0,0.0,1.0,12,0.000000,0.083333,1.0,0.083333,BELÉM NOVO - PORTO ALEGRE - RS
78,SIXE,0.0,0.0,1.0,1.0,0.0,17,0.058824,0.000000,1.0,0.058824,AEROCLUBE DE ELDORADO DO SUL - ELDORADO DO SUL...
5,SBBH,1.0,5.0,1.0,2.0,5.0,177,0.011299,0.028249,7.0,0.039548,PAMPULHA - CARLOS DRUMMOND DE ANDRADE - BELO H...
82,SSCN,1.0,1.0,0.0,1.0,1.0,68,0.014706,0.014706,2.0,0.029412,CANELA - CANELA - RS
83,SSLT,0.0,1.0,0.0,0.0,1.0,66,0.000000,0.015152,1.0,0.015152,GAUDÊNCIO MACHADO RAMOS - ALEGRETE - RS
85,SWBC,0.0,1.0,0.0,0.0,1.0,80,0.000000,0.012500,1.0,0.012500,BARCELOS - BARCELOS - AM


Nos resultados de taxa por aeroporto, observa-se que o ranking muda quando a exposição entra no denominador: aeroportos de grande movimento deixam de aparecer automaticamente como os de maior risco relativo. Isso confirma o efeito de base rate discutido na introdução e sustenta a validação das hipóteses H1 e H3. A seção seguinte consolida essa leitura com visualização em escala log-log e limiares por percentil.

In [88]:
# ppm = ocorrências por 1 milhão de movimentos (taxa × 1e6) — linhas por aeroporto (ordenados por movimentação), 2023 e 2024
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _prep_linhas_ppm(df_taxa):
    d = df_taxa[df_taxa["movimentacao_total"] >= 100].sort_values(
        "movimentacao_total", ascending=True
    )
    d = d.reset_index(drop=True)
    d["ordem"] = np.arange(1, len(d) + 1)
    d["ppm_total"] = d["taxa_total"] * 1_000_000
    d["ppm_grave"] = d["taxa_grave"] * 1_000_000
    d["ppm_incidente"] = d["taxa_incidente"] * 1_000_000
    return d


_hover_total = (
    "<b>%{text}</b> (ICAO %{customdata[0]})<br>"
    "Movimentos: %{customdata[1]:,.0f}<br>"
    "Taxa total: %{y:,.1f} ppm<extra></extra>"
)
_hover_grave = (
    "<b>%{text}</b> (ICAO %{customdata[0]})<br>"
    "Movimentos: %{customdata[1]:,.0f}<br>"
    "Taxa grave: %{y:,.1f} ppm<extra></extra>"
)
_hover_inc = (
    "<b>%{text}</b> (ICAO %{customdata[0]})<br>"
    "Movimentos: %{customdata[1]:,.0f}<br>"
    "Taxa incidente: %{y:,.1f} ppm<extra></extra>"
)

fig_taxa_mov = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("2023", "2024"),
    horizontal_spacing=0.07,
    shared_yaxes=True,
)

for _col, (_df_src, _show_leg) in enumerate(
    (
        (df_taxa_aeroporto_2023, True),
        (df_taxa_aeroporto_2024, False),
    ),
    start=1,
):
    _d = _prep_linhas_ppm(_df_src)
    _cd = np.stack(
        [
            _d["aeroporto"].astype(str),
            _d["movimentacao_total"].to_numpy(),
        ],
        axis=-1,
    )
    fig_taxa_mov.add_trace(
        go.Scatter(
            x=_d["ordem"],
            y=_d["ppm_total"],
            mode="lines",
            name="Taxa total (ppm)",
            text=_d["nome_aeroporto"],
            customdata=_cd,
            line=dict(color="#636EFA", width=2.8),
            hovertemplate=_hover_total,
            showlegend=_show_leg,
            legendgroup="tot",
        ),
        row=1,
        col=_col,
    )
    fig_taxa_mov.add_trace(
        go.Scatter(
            x=_d["ordem"],
            y=_d["ppm_grave"],
            mode="lines",
            name="Taxa grave (ppm)",
            text=_d["nome_aeroporto"],
            customdata=_cd,
            line=dict(color="rgba(239,85,59,0.45)", width=1.8),
            hovertemplate=_hover_grave,
            showlegend=_show_leg,
            legendgroup="grave",
        ),
        row=1,
        col=_col,
    )
    fig_taxa_mov.add_trace(
        go.Scatter(
            x=_d["ordem"],
            y=_d["ppm_incidente"],
            mode="lines",
            name="Taxa incidente (ppm)",
            text=_d["nome_aeroporto"],
            customdata=_cd,
            line=dict(color="rgba(39,211,245,0.45)", width=1.8),
            hovertemplate=_hover_inc,
            showlegend=_show_leg,
            legendgroup="inc",
        ),
        row=1,
        col=_col,
    )

fig_taxa_mov.update_xaxes(title_text="Ordem do aeroporto (menor → maior tráfego)", row=1, col=1)
fig_taxa_mov.update_xaxes(title_text="", row=1, col=2)
fig_taxa_mov.update_yaxes(title_text="Taxa (ocorrências / milhão de movimentos)", row=1, col=1)

fig_taxa_mov.update_layout(
    title="Taxas por aeroporto (ppm) — 2023 e 2024, movimentação ≥ 100 (ordenado por movimentação crescente)",
    title_x=0.05,
    yaxis=dict(rangemode="tozero", tickformat=",.0f"),
    legend=dict(
        orientation="h",
        y=1.14,
        yanchor="bottom",
        x=0.5,
        xanchor="center",
    ),
    height=540,
    margin=dict(l=72, r=48, t=88, b=56),
)

fig_taxa_mov.show()


Usando o mesmo eixo de aeroportos de menor para maior porte, observamos que no intervalo a movimentação das linhas mínima. Utilizando uma escala ppm (partes por milhão) se torna visível a forte heterogeneidade do risco relativo quando a exposição cresce. A série principal em azul (taxa total) combina todas as classificações e costuma situar-se acima das demais por construção; as séries para taxa grave e taxa incidente confirmam que incidentes leves concentram a maior parte da massa de ocorrências, enquanto eventos graves permanecem em patamares inferiores mas com picos pontuais em aeroportos de menor tráfego. A leitura reforça que comparar aeroportos apenas pelo numerador absoluto distorce a conclusão: mesmo em ppm, a variabilidade na cauda de baixa movimentação exige cautela interpretativa (denominadores pequenos amplificam taxas), alinhando-se ao argumento de normalizar por exposição adotado ao longo do estudo.

In [56]:
# Estatísticas descritivas das taxas por aeroporto (por ano)
_cols = ['taxa_total', 'taxa_incidente', 'taxa_grave']


def _resumo_taxas(df, ano):
    linhas = []
    for col in _cols:
        s = df[col].dropna()
        moda = s.mode()
        linhas.append(
            {
                'ano': ano,
                'variavel': col,
                'media': s.mean(),
                'mediana': s.median(),
                'moda': moda.iloc[0] if len(moda) else np.nan,
                'amplitude': s.max() - s.min(),
                'desvio_padrao': s.std(),
            }
        )
    return pd.DataFrame(linhas)


display(
    pd.concat(
        [
            _resumo_taxas(df_taxa_aeroporto_2023, 2023),
            _resumo_taxas(df_taxa_aeroporto_2024, 2024),
        ],
        ignore_index=True,
    )
)

,ano,variavel,media,mediana,moda,amplitude,desvio_padrao
0,2023,taxa_total,0.083045,0.000959,0.000059,4.999941,0.550379
1,2023,taxa_incidente,0.063486,0.000701,0.000000,4.000000,0.440277
2,2023,taxa_grave,0.019559,0.000000,0.000000,1.000000,0.113839
3,2024,taxa_total,0.029619,0.001348,0.000382,1.499618,0.166780
4,2024,taxa_incidente,0.015416,0.001080,0.000000,1.000000,0.106019
5,2024,taxa_grave,0.014203,0.000000,0.000000,0.500000,0.075960


## Análise dos resultados


In [57]:
#scatter log-log: movimentos x ocorrencias por aeroporto com limiares de percentil
#percentil 50 = aeroporto tipico; percentil 75 = limiar de risco elevado
#pontos acima da linha P75 sao aeroportos com taxa de ocorrencia acima do esperado para seu volume

import plotly.graph_objects as go
import numpy as np

def _scatter_loglog_taxa(df_2023, df_2024):
    df_all = pd.concat([
        df_2023.assign(ano='2023'),
        df_2024.assign(ano='2024')
    ], ignore_index=True)

    if 'nome_aeroporto' not in df_all.columns:
        df_all['nome_aeroporto'] = df_all['aeroporto']

    p50 = df_all['taxa_total'].quantile(0.50)
    p75 = df_all['taxa_total'].quantile(0.75)

    x_min = df_all['movimentacao_total'].min()
    x_max = df_all['movimentacao_total'].max()
    x_range = np.logspace(np.log10(x_min), np.log10(x_max), 200)

    fig = go.Figure()

    for ano, cor in [('2023', '#636EFA'), ('2024', '#EF553B')]:
        sub = df_all[df_all['ano'] == ano]
        fig.add_trace(go.Scatter(
            x=sub['movimentacao_total'],
            y=sub['total_ocorrencias'],
            mode='markers',
            name=ano,
            text=sub['nome_aeroporto'],
            customdata=sub['aeroporto'],
            hovertemplate='<b>%{text}</b> (%{customdata})<br>Movimentos: %{x:,.0f}<br>Ocorrências: %{y}<extra></extra>',
            marker=dict(color=cor, opacity=0.7, size=7)
        ))

    fig.add_trace(go.Scatter(
        x=x_range,
        y=p50 * x_range,
        mode='lines',
        name=f'Percentil 50 — taxa típica ({p50:.5f})',
        line=dict(color='gray', dash='dash', width=1.5)
    ))

    fig.add_trace(go.Scatter(
        x=x_range,
        y=p75 * x_range,
        mode='lines',
        name=f'Percentil 75 — risco elevado ({p75:.5f})',
        line=dict(color='orange', dash='dot', width=2)
    ))

    fig.update_layout(
        title='Movimentos vs Ocorrências por Aeroporto (escala log-log)',
        xaxis=dict(title='Total de Movimentos (log)', type='log'),
        yaxis=dict(title='Total de Ocorrências (log)', type='log'),
        legend=dict(orientation='h', y=1.08, x=0),
        height=560
    )

    return fig

fig_scatter_loglog = _scatter_loglog_taxa(df_taxa_aeroporto_2023, df_taxa_aeroporto_2024)
fig_scatter_loglog.show()

### Escala log-log
A correlação linear entre `total_movimentos` e `total_ocorrencias` tende a ser alta por construção, porque ambas as variáveis crescem com o porte do aeroporto. Esse resultado é informativo para volume, mas pouco útil para risco relativo: ele não separa efeito de escala (aeroporto grande) de desempenho de segurança (taxa por movimento).

A visualização em escala log-log é mais adequada por três motivos. Primeiro, reduz a influência de outliers de tráfego muito alto e permite comparar pequenos e grandes aeroportos no mesmo painel. Segundo, transforma relações de potência em tendências lineares, facilitando avaliar se há proporcionalidade entre exposição e ocorrências. Terceiro, torna o **desvio vertical** em relação às linhas de referência diretamente interpretável como excesso ou déficit de taxa para um mesmo nível de movimento.

No gráfico, observa-se um padrão compatível com comportamento sublinear agregado: ao aumentar o volume de movimentos, o crescimento das ocorrências não mantém a mesma razão em todos os níveis de tráfego. Em termos práticos, há aeroportos de baixo volume com taxas relativamente altas (acima das linhas de referência), enquanto hubs de alto movimento permanecem majoritariamente abaixo desses limiares. Isso reforça a leitura de que contagem absoluta não é um bom proxy de risco e dá suporte empírico à H1.

### Percentis como limiar de referência

A escolha de percentis como linha de corte não é apenas uma alternativa visual à média: ela define um critério de comparação mais robusto para distribuições assimétricas e com cauda longa, como é o caso das taxas por aeroporto. Em distribuições desse tipo, a média é instável a poucos valores extremos e tende a deslocar artificialmente o referencial para cima.

Ao usar o **P50**, adotamos uma referência de comportamento típico do sistema (mediana), menos sensível a extremos. Já o **P75** funciona como faixa de atenção operacional: aeroportos acima desse limiar não são necessariamente "inseguros", mas exibem taxa superior à de pelo menos 75% dos pares observados e, portanto, merecem análise complementar (perfil de operação, contexto local, tipo de tráfego e qualidade de reporte).

Outro ponto importante é que os percentis preservam comparabilidade entre aeroportos de portes diferentes sem impor um modelo paramétrico forte. Em vez de concluir por ranking absoluto, o critério por quartis permite priorização: abaixo de P50 (melhor desempenho relativo), entre P50 e P75 (faixa intermediária) e acima de P75 (prioridade investigativa). Essa interpretação evita leituras simplistas e torna o resultado mais acionável para gestão de risco.

### Modelo de contagem com offset (Poisson e Binomial Negativa)

Nesta seção, a modelagem é feita com `offset(log(movimentos))` para incorporar a exposição operacional de cada aeroporto. Com isso, os resultados passam a refletir taxa por movimento, reduzindo viés de comparação por porte.

In [60]:
# base consolidada para modelos de contagem com offset
import numpy as np
import pandas as pd

try:
    import statsmodels.api as sm
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "statsmodels não encontrado. Instale com: pip install statsmodels"
    ) from e

# consolidação por aeroporto (2023-2024)
df_count_model = pd.concat([
    df_taxa_aeroporto_2023.assign(ano='2023'),
    df_taxa_aeroporto_2024.assign(ano='2024')
], ignore_index=True)

df_count_model = (
    df_count_model.groupby('aeroporto', as_index=False)
    .agg({
        'ACIDENTE': 'sum',
        'total_ocorrencias': 'sum',
        'movimentacao_total': 'sum',
        'nome_aeroporto': 'first',
    })
    .rename(columns={
        'movimentacao_total': 'movimentos',
        'ACIDENTE': 'acidentes',
    })
)

# variáveis do modelo
df_count_model = df_count_model[df_count_model['movimentos'] > 0].copy()
df_count_model['log_movimentos'] = np.log(df_count_model['movimentos'])

display(df_count_model[['aeroporto', 'nome_aeroporto', 'acidentes', 'total_ocorrencias', 'movimentos', 'log_movimentos']].head(10))
print(f"Amostra final do modelo: {len(df_count_model)} aeroportos")

,aeroporto,nome_aeroporto,acidentes,total_ocorrencias,movimentos,log_movimentos
0,SBAE,BAURU/AREALVA (SJTC*) - AREALVA - SP,1.0,2.0,2547,7.842671
1,SBAR,SANTA MARIA - ARACAJU - SE,0.0,12.0,19660,9.886341
2,SBAT,PILOTO OSVALDO MARQUES DIAS - ALTA FLORESTA - MT,1.0,4.0,509,6.232448
3,SBAU,ESTADUAL DARIO GUARITA - ARAÇATUBA - SP,0.0,4.0,3131,8.049108
4,SBBE,INTERNACIONAL DE BELÉM/VAL DE CANS/JÚLIO CEZAR...,1.0,45.0,62852,11.048538
5,SBBG,COMANDANTE GUSTAVO KRAEMER - BAGÉ - RS,0.0,1.0,254,5.537334
6,SBBH,PAMPULHA - CARLOS DRUMMOND DE ANDRADE - BELO H...,1.0,16.0,359,5.883322
7,SBBI,BACACHERI - CURITIBA - PR,0.0,5.0,1,0.000000
8,SBBP,ESTADUAL ARTHUR SIQUEIRA - BRAGANÇA PAULISTA - SP,2.0,7.0,6,1.791759
9,SBBR,PRESIDENTE JUSCELINO KUBITSCHEK - BRASÍLIA - DF,0.0,112.0,219636,12.299727


Amostra final do modelo: 106 aeroportos


In [61]:
# modelo Poisson com offset(log_movimentos)
# alvo principal: acidentes (evento raro)

y = df_count_model['acidentes'].to_numpy()
X = np.ones((len(df_count_model), 1))  # intercepto apenas

pois_res = sm.GLM(
    y,
    X,
    family=sm.families.Poisson(),
    offset=df_count_model['log_movimentos'].to_numpy(),
).fit()

mu_hat_pois = pois_res.fittedvalues
pearson_chi2 = np.sum(((y - mu_hat_pois) ** 2) / np.clip(mu_hat_pois, 1e-12, None))
pearson_ratio = pearson_chi2 / pois_res.df_resid if pois_res.df_resid > 0 else np.nan
dev_ratio = pois_res.deviance / pois_res.df_resid if pois_res.df_resid > 0 else np.nan

poisson_summary = pd.DataFrame([
    {
        'modelo': 'Poisson + offset(log_movimentos)',
        'intercepto': float(pois_res.params[0]),
        'se_intercepto': float(pois_res.bse[0]),
        'pvalor_intercepto': float(pois_res.pvalues[0]),
        'ic95_low': float(pois_res.conf_int()[0, 0]),
        'ic95_high': float(pois_res.conf_int()[0, 1]),
        'log_likelihood': float(pois_res.llf),
        'AIC': float(pois_res.aic),
        'deviance_ratio': float(dev_ratio),
        'pearson_ratio': float(pearson_ratio),
    }
])

display(poisson_summary)

,modelo,intercepto,se_intercepto,pvalor_intercepto,ic95_low,ic95_high,log_likelihood,AIC,deviance_ratio,pearson_ratio
0,Poisson + offset(log_movimentos),-11.412486,0.164399,0.0,-11.734702,-11.09027,-201.544747,405.089495,3.242052,1568.957973


Critério adotado: se `deviance_ratio` e/ou `pearson_ratio` forem claramente superiores a 1, há evidência de sobredispersão e a Binomial Negativa tende a ser mais adequada que o Poisson.

In [62]:
# modelo Binomial Negativa com offset(log_movimentos)
# versão estável: estima alpha por momentos a partir do Poisson e ajusta GLM-NB

# estimador de momentos para alpha (var = mu + alpha * mu^2)
num = np.sum((y - mu_hat_pois) ** 2 - y)
den = np.sum(np.clip(mu_hat_pois, 1e-12, None) ** 2)
alpha_hat = float(max(num / den, 1e-8)) if den > 0 else 1e-8

nb_glm_res = sm.GLM(
    y,
    X,
    family=sm.families.NegativeBinomial(alpha=alpha_hat),
    offset=df_count_model['log_movimentos'].to_numpy(),
).fit()

nb_summary = pd.DataFrame([
    {
        'modelo': 'GLM NegBin + offset(log_movimentos)',
        'intercepto': float(nb_glm_res.params[0]),
        'se_intercepto': float(nb_glm_res.bse[0]),
        'pvalor_intercepto': float(nb_glm_res.pvalues[0]),
        'ic95_low': float(nb_glm_res.conf_int()[0, 0]),
        'ic95_high': float(nb_glm_res.conf_int()[0, 1]),
        'log_likelihood': float(nb_glm_res.llf),
        'AIC': float(nb_glm_res.aic),
        'alpha_estimado_momentos': alpha_hat,
    }
])

display(nb_summary)

,modelo,intercepto,se_intercepto,pvalor_intercepto,ic95_low,ic95_high,log_likelihood,AIC,alpha_estimado_momentos
0,GLM NegBin + offset(log_movimentos),-10.071074,0.18436,0.0,-10.432414,-9.709734,-173.219199,348.438398,0.978226


In [63]:
# comparação Poisson vs NegBin
compare_df = pd.DataFrame([
    {
        'modelo': 'Poisson + offset(log_movimentos)',
        'AIC': float(pois_res.aic),
        'log_likelihood': float(pois_res.llf),
        'deviance_ratio': float(dev_ratio),
        'pearson_ratio': float(pearson_ratio),
        'intercepto': float(pois_res.params[0]),
    },
    {
        'modelo': 'GLM NegBin + offset(log_movimentos)',
        'AIC': float(nb_glm_res.aic),
        'log_likelihood': float(nb_glm_res.llf),
        'deviance_ratio': np.nan,
        'pearson_ratio': np.nan,
        'intercepto': float(nb_glm_res.params[0]),
    },
]).sort_values('AIC')

display(compare_df)

if (dev_ratio > 1.5) or (pearson_ratio > 1.5):
    print('Decisão sugerida: NegBin (indício de sobredispersão no Poisson).')
else:
    print('Decisão sugerida: Poisson pode ser suficiente para esta amostra.')

,modelo,AIC,log_likelihood,deviance_ratio,pearson_ratio,intercepto
1,GLM NegBin + offset(log_movimentos),348.438398,-173.219199,NaN,NaN,-10.071074
0,Poisson + offset(log_movimentos),405.089495,-201.544747,3.242052,1568.957973,-11.412486


Decisão sugerida: NegBin (indício de sobredispersão no Poisson).


Os resultados indicam Poisson com sobredispersão relevante e melhor ajuste da Binomial Negativa (AIC menor e log-likelihood maior). Assim, a Binomial Negativa com offset é adotada como modelo principal de inferência neste estudo.

In [64]:
# interpretação prática do intercepto: taxa base por movimento
import numpy as np

rate_pois = float(np.exp(pois_res.params[0]))
rate_nb = float(np.exp(nb_glm_res.params[0]))

taxa_interpretavel = pd.DataFrame([
    {
        'modelo': 'Poisson + offset(log_movimentos)',
        'taxa_base_por_movimento': rate_pois,
        'eventos_por_100k_movimentos': rate_pois * 100000,
    },
    {
        'modelo': 'GLM NegBin + offset(log_movimentos)',
        'taxa_base_por_movimento': rate_nb,
        'eventos_por_100k_movimentos': rate_nb * 100000,
    },
])

display(taxa_interpretavel)

,modelo,taxa_base_por_movimento,eventos_por_100k_movimentos
0,Poisson + offset(log_movimentos),0.000011,1.105657
1,GLM NegBin + offset(log_movimentos),0.000042,4.228517


In [65]:
# previsões esperadas para diferentes níveis de exposição
exposicoes = np.array([10000, 50000, 100000, 250000], dtype=float)

pred_df = pd.DataFrame({
    'movimentos': exposicoes.astype(int),
})
pred_df['esperado_poisson'] = np.exp(pois_res.params[0]) * pred_df['movimentos']
pred_df['esperado_negbin'] = np.exp(nb_glm_res.params[0]) * pred_df['movimentos']

display(pred_df)

,movimentos,esperado_poisson,esperado_negbin
0,10000,0.110566,0.422852
1,50000,0.552828,2.114259
2,100000,1.105657,4.228517
3,250000,2.764142,10.571293


In [66]:
# robustez: repetir Poisson/NegBin usando total_ocorrencias como alvo

y_total = df_count_model['total_ocorrencias'].to_numpy()

pois_total_res = sm.GLM(
    y_total,
    X,
    family=sm.families.Poisson(),
    offset=df_count_model['log_movimentos'].to_numpy(),
).fit()

mu_hat_pois_total = pois_total_res.fittedvalues
pearson_chi2_total = np.sum(((y_total - mu_hat_pois_total) ** 2) / np.clip(mu_hat_pois_total, 1e-12, None))
pearson_ratio_total = pearson_chi2_total / pois_total_res.df_resid if pois_total_res.df_resid > 0 else np.nan
dev_ratio_total = pois_total_res.deviance / pois_total_res.df_resid if pois_total_res.df_resid > 0 else np.nan

num_total = np.sum((y_total - mu_hat_pois_total) ** 2 - y_total)
den_total = np.sum(np.clip(mu_hat_pois_total, 1e-12, None) ** 2)
alpha_total = float(max(num_total / den_total, 1e-8)) if den_total > 0 else 1e-8

nb_total_res = sm.GLM(
    y_total,
    X,
    family=sm.families.NegativeBinomial(alpha=alpha_total),
    offset=df_count_model['log_movimentos'].to_numpy(),
).fit()

robustez_df = pd.DataFrame([
    {
        'alvo': 'total_ocorrencias',
        'modelo': 'Poisson + offset',
        'AIC': float(pois_total_res.aic),
        'log_likelihood': float(pois_total_res.llf),
        'deviance_ratio': float(dev_ratio_total),
        'pearson_ratio': float(pearson_ratio_total),
    },
    {
        'alvo': 'total_ocorrencias',
        'modelo': 'GLM NegBin + offset',
        'AIC': float(nb_total_res.aic),
        'log_likelihood': float(nb_total_res.llf),
        'deviance_ratio': np.nan,
        'pearson_ratio': np.nan,
        'alpha_estimado_momentos': float(alpha_total),
    },
]).sort_values('AIC')

display(robustez_df)

,alvo,modelo,AIC,log_likelihood,deviance_ratio,pearson_ratio,alpha_estimado_momentos
1,total_ocorrencias,GLM NegBin + offset,982.872031,-490.436016,NaN,NaN,0.141464
0,total_ocorrencias,Poisson + offset,1417.109958,-707.554979,9.633248,544.10827,NaN


Em síntese, percentis e OLS são mantidos como apoio descritivo e exploratório. A inferência principal passa para a Binomial Negativa com offset, devido à sobredispersão no Poisson e ao ganho consistente de ajuste. A análise de robustez com `total_ocorrencias` mantém essa conclusão.

A comparação Poisson vs Binomial Negativa complementa a leitura dos percentis e do OLS. Diante de sobredispersão, a Binomial Negativa oferece inferência mais estável para contagens raras. Isso reforça a conclusão de que segurança operacional deve ser avaliada por exposição normalizada e por modelos de contagem adequados.

In [67]:
media_acidentes = df_acidentes_cidade['contagem'].mean()
media_incidentes = df_incidentes_cidade['contagem'].mean()
media_incidentes_graves = df_incidentes_graves_cidade['contagem'].mean()

mediana_acidentes = df_acidentes_cidade['contagem'].median()
mediana_incidentes = df_incidentes_cidade['contagem'].median()
mediana_incidentes_graves = df_incidentes_graves_cidade['contagem'].median()

std_acidentes = df_acidentes_cidade['contagem'].std()
std_incidentes = df_incidentes_cidade['contagem'].std()
std_incidentes_graves = df_incidentes_graves_cidade['contagem'].std()

print(f"Média de acidentes:                 {media_acidentes:.2f}")
print(f"Mediana de acidentes:               {mediana_acidentes:.2f}")
print(f"Desvio padrão de acidentes:         {std_acidentes:.2f}")

print(f"Média de incidentes:                {media_incidentes:.2f}")
print(f"Mediana de incidentes:              {mediana_incidentes:.2f}")
print(f"Desvio padrão de incidentes:        {std_incidentes:.2f}")

print(f"Média de incidentes graves:         {media_incidentes_graves:.2f}")
print(f"Mediana de incidentes graves:       {mediana_incidentes_graves:.2f}")
print(f"Desvio padrão de incidentes graves: {std_incidentes_graves:.2f}")

Média de acidentes:                 1.30
Mediana de acidentes:               1.00
Desvio padrão de acidentes:         0.62
Média de incidentes:                17.75
Mediana de incidentes:              2.00
Desvio padrão de incidentes:        49.37
Média de incidentes graves:         1.21
Mediana de incidentes graves:       1.00
Desvio padrão de incidentes graves: 0.58


In [68]:
#voos por ocorrencia nos 5 aeroportos com maior volume de movimentos (escala inversa)
#leitura: a cada X voos, ocorre 1 evento — quanto maior o numero, mais seguro o aeroporto
#contraponto ao ranking por taxa: mostra que os grandes hubs tem numeros muito mais altos (mais seguros por voo)

def _bar_hubs_inverso(df, ano):
    top = df.nlargest(5, 'movimentacao_total').copy()
    if 'nome_aeroporto' not in top.columns:
        top['nome_aeroporto'] = top['aeroporto']

    top['voos_por_grave']     = (1 / top['taxa_grave']).replace([np.inf], pd.NA)
    top['voos_por_incidente'] = (1 / top['taxa_incidente']).replace([np.inf], pd.NA)

    df_melted = top.melt(
        id_vars=['aeroporto', 'nome_aeroporto', 'movimentacao_total'],
        value_vars=['voos_por_grave', 'voos_por_incidente'],
        var_name='tipo',
        value_name='voos_por_ocorrencia'
    )
    df_melted['tipo'] = df_melted['tipo'].map({
        'voos_por_grave':     'Graves (Acidentes + Inc. Graves)',
        'voos_por_incidente': 'Incidentes',
    })
    df_melted['rotulo'] = df_melted['voos_por_ocorrencia'].apply(
        lambda x: f'1 em {x:,.0f} voos' if pd.notna(x) else 'sem ocorrências'
    )

    order = top.sort_values('movimentacao_total', ascending=True)['nome_aeroporto'].tolist()

    fig = ptex.bar(
        df_melted.dropna(subset=['voos_por_ocorrencia']),
        x='voos_por_ocorrencia',
        y='nome_aeroporto',
        color='tipo',
        orientation='h',
        barmode='group',
        text='rotulo',
        title=f'{ano} — Intervalo médio entre ocorrências (5 aeroportos mais movimentados)',
        labels={
            'voos_por_ocorrencia': 'Voos por ocorrência  ·  maior = mais seguro',
            'nome_aeroporto': 'Aeroporto',
            'tipo': 'Tipo de ocorrência',
        },
        color_discrete_map={
            'Graves (Acidentes + Inc. Graves)': '#EF553B',
            'Incidentes': '#27D3F5',
        },
        category_orders={'nome_aeroporto': order},
        hover_data={'aeroporto': True, 'movimentacao_total': ':,.0f'},
        height=420
    )
    fig.update_traces(textposition='outside', cliponaxis=False)
    fig.update_layout(
        legend=dict(orientation='h', y=1.12, x=0),
        margin=dict(r=120),
    )
    return fig

fig_hubs_2023 = _bar_hubs_inverso(df_taxa_aeroporto_2023, '2023')
fig_hubs_2024 = _bar_hubs_inverso(df_taxa_aeroporto_2024, '2024')
fig_hubs_2023.show()
fig_hubs_2024.show()

### Comparando o risco operacional de grandes aeroportos frente aos aeroportos mais seguros

Os dois gráficos superiores mostram que aeroportos com maior movimentação não apresentam, necessariamente, pior desempenho relativo quando a métrica é normalizada por exposição. Em outras palavras, alto volume de operações aumenta o número absoluto de eventos, mas não implica, por si só, maior taxa de ocorrência por voo.

Nos dois gráficos inferiores, ao observar os aeroportos com menor taxa, vemos que a diferença principal está no intervalo médio entre ocorrências (voos por evento), e não apenas no tamanho do aeroporto. Esse contraste evidencia que o risco relativo é mais bem explicado por taxa normalizada do que por contagem bruta.

Analiticamente, essa comparação fortalece a H3: métricas absolutas tendem a favorecer interpretações enviesadas pelo porte operacional, enquanto a normalização por movimentação permite uma leitura mais justa e comparável entre aeroportos de perfis distintos. Como implicação prática, a priorização de ações de segurança deve considerar desempenho relativo (taxa) e contexto operacional, e não somente volume total de ocorrências.

In [69]:
#top 5 aeroportos mais seguros (maior intervalo entre ocorrencias, escala inversa)
#exclui aeroportos com taxa_total == 0 (inverso indefinido) antes de ordenar

def _bar_safest(df, ano):
    candidatos = df[df['taxa_total'] > 0].copy()
    top = candidatos.nsmallest(5, 'taxa_total').copy()

    if 'nome_aeroporto' not in top.columns:
        top['nome_aeroporto'] = top['aeroporto']

    top['voos_por_grave']     = (1 / top['taxa_grave']).replace([np.inf], pd.NA)
    top['voos_por_incidente'] = (1 / top['taxa_incidente']).replace([np.inf], pd.NA)

    df_melted = top.melt(
        id_vars=['aeroporto', 'nome_aeroporto', 'movimentacao_total'],
        value_vars=['voos_por_grave', 'voos_por_incidente'],
        var_name='tipo',
        value_name='voos_por_ocorrencia'
    )
    df_melted['tipo'] = df_melted['tipo'].map({
        'voos_por_grave':     'Graves (Acidentes + Inc. Graves)',
        'voos_por_incidente': 'Incidentes',
    })
    df_melted['rotulo'] = df_melted['voos_por_ocorrencia'].apply(
        lambda x: f'1 em {x:,.0f} voos' if pd.notna(x) else 'sem ocorrências'
    )

    order = top.sort_values('taxa_total', ascending=False)['nome_aeroporto'].tolist()

    fig = ptex.bar(
        df_melted.dropna(subset=['voos_por_ocorrencia']),
        x='voos_por_ocorrencia',
        y='nome_aeroporto',
        color='tipo',
        orientation='h',
        barmode='group',
        text='rotulo',
        title=f'{ano} — Top 5 Aeroportos mais seguros por voo (menor taxa de ocorrência)',
        labels={
            'voos_por_ocorrencia': 'Voos por ocorrência  ·  maior = mais seguro',
            'nome_aeroporto': 'Aeroporto',
            'tipo': 'Tipo de ocorrência',
        },
        color_discrete_map={
            'Graves (Acidentes + Inc. Graves)': '#EF553B',
            'Incidentes': '#27D3F5',
        },
        category_orders={'nome_aeroporto': order},
        hover_data={'aeroporto': True, 'movimentacao_total': ':,.0f'},
        height=420
    )
    fig.update_traces(textposition='outside', cliponaxis=False)
    fig.update_layout(
        legend=dict(orientation='h', y=1.12, x=0),
        margin=dict(r=120),
    )
    return fig

fig_safest_2023 = _bar_safest(df_taxa_aeroporto_2023, '2023')
fig_safest_2024 = _bar_safest(df_taxa_aeroporto_2024, '2024')
fig_safest_2023.show()
fig_safest_2024.show()

## Limitações do Estudo
As maiores limitações do estudo estão no fato de que o VRA é uma amostra uniforme do movimento aeroviário, estando o estudo sujeito a um viés de exposição da aviação civil. Também as simplificações do estudo consideram um mapeamento heurístico para ligar a fase de voo ao aeroporto onde a ocorrência teve origem. Um número de ocorrências não pôde ser mapeada a um aeroporto específico devido a natureza ambígua da fase (taxi, manobra, estacionamento, outra fase, etc). Outro tipo de simplificação.

Aqui tratamos de uma métrica direta e objetiva (ocorrência por milhares de voos). Porém, no panorama de segurança operacional, também podem ser considerados outros fatores como manutenção da aeronave, condições meteorológicas, experiência de piloto e volume de passageiros.

### Limitação de interpretação por tipo de operação

O gráfico de taxa de acidentes por tipo de operação indica que a aviação civil regular apresenta a menor taxa relativa entre os grupos comparados no período analisado. Esse resultado é compatível com as análises anteriores, que se concentraram em aeroportos de grande porte, nos quais a operação regular é predominante e os processos de gestão de segurança tendem a ser mais maduros.

Ao mesmo tempo, essa diferença não deve ser interpretada como evidência isolada de “maior segurança intrínseca” de um tipo de operação. Parte relevante desse comportamento é estrutural e esperada: a aviação civil regular opera sob um marco regulatório mais rígido, maior padronização operacional, maior frequência de auditorias e rotinas mais consolidadas de monitoramento de risco. Portanto, a comparação entre tipos de operação deve considerar diferenças de contexto regulatório e operacional para evitar conclusões causais simplificadas.

Apesar de representar uma limitação interpretativa neste MVP, esse ponto abre uma oportunidade relevante para estudos futuros: avaliar a taxa de ocorrência com estratificação adicional por tipo de operação, perfil de missão, nível de regulação e ambiente operacional, permitindo estimativas mais comparáveis e inferências mais robustas sobre risco relativo.

In [70]:
# taxa de acidentes por tipo de operação (2023-2024)
# representação escolhida: dot plot horizontal (taxa no eixo X + tamanho da bolha = volume)

# 1) identifica coluna de tipo de operação disponível
candidatas_tipo_operacao = [
    'aeronave_tipo_operacao',
    'aeronave_tipo_veiculo',
    'aeronave_tipo_voo',
    'ocorrencia_tipo_operacao',
]

col_tipo_operacao = next((c for c in candidatas_tipo_operacao if c in df_aeronave.columns), None)
if col_tipo_operacao is None:
    raise ValueError('Nenhuma coluna de tipo de operação foi encontrada em df_aeronave.')

# 2) base analítica: classificação da ocorrência + tipo de operação
base_tx = (
    df_ocorrencia[['codigo_ocorrencia2', 'ocorrencia_classificacao', 'ocorrencia_dia']]
    .merge(
        df_aeronave[['codigo_ocorrencia2', col_tipo_operacao]],
        on='codigo_ocorrencia2',
        how='inner'
    )
    .copy()
)

base_tx = filter_by_year(base_tx, 'ocorrencia_dia', [2023, 2024])
base_tx = base_tx[base_tx[col_tipo_operacao].notna()]

# 3) calcula taxa de acidentes por tipo de operação
resumo_tipo = (
    base_tx.groupby(col_tipo_operacao)
    .agg(
        total_ocorrencias=('ocorrencia_classificacao', 'size'),
        acidentes=('ocorrencia_classificacao', lambda s: (s == 'ACIDENTE').sum()),
    )
    .reset_index()
)

resumo_tipo['taxa_acidente'] = resumo_tipo['acidentes'] / resumo_tipo['total_ocorrencias']
resumo_tipo['taxa_pct'] = (100 * resumo_tipo['taxa_acidente']).round(2)

# filtro mínimo para reduzir ruído de categorias com amostra muito pequena
resumo_tipo = resumo_tipo[resumo_tipo['total_ocorrencias'] >= 10].copy()
resumo_tipo = resumo_tipo.sort_values('taxa_acidente', ascending=True)

# 4) dot plot: x = taxa, y = tipo, tamanho = volume (contexto de confiabilidade)
fig_taxa_tipo_operacao = ptex.scatter(
    resumo_tipo,
    x='taxa_acidente',
    y=col_tipo_operacao,
    size='total_ocorrencias',
    color='taxa_acidente',
    color_continuous_scale='OrRd',
    labels={
        'taxa_acidente': 'Taxa de acidentes',
        col_tipo_operacao: 'Tipo de operação',
        'total_ocorrencias': 'Total de ocorrências',
    },
    title='Taxa de acidentes por tipo de operação (2023-2024)'
)

fig_taxa_tipo_operacao.update_traces(
    text=resumo_tipo['taxa_pct'].astype(str) + '%',
    textposition='middle right',
    marker=dict(opacity=0.85, line=dict(width=0.5, color='DarkSlateGrey')),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Taxa de acidentes: %{x:.2%}<br>'
        'Total de ocorrências: %{marker.size:,}<br>'
        '<extra></extra>'
    ),
)

fig_taxa_tipo_operacao.update_layout(
    height=520,
    xaxis_tickformat='.0%',
    coloraxis_colorbar_title='Taxa',
)

fig_taxa_tipo_operacao.show()



# Bibliografia
INTERNATIONAL CIVIL AVIATION ORGANIZATION. Safety Management Manual (SMM). 4. ed. Montreal: ICAO, 2018.

INTERNATIONAL AIR TRANSPORT ASSOCIATION. Safety Report 2023. Montreal: IATA, 2023

CENTRO DE INVESTIGAÇÃO E PREVENÇÃO DE ACIDENTES AERONÁUTICOS. NSCA 3-13: Protocolos de Investigação de Ocorrências Aeronáuticas. Brasília: CENIPA, 2018.

STOLZER, Alan J.; HALFORD, Carl D.; GOGLIA, John J. Introduction to Safety Management Systems. 2. ed. Boca Raton: CRC Press, 2016.

INTERNATIONAL CIVIL AVIATION ORGANIZATION. Safety Report 2023. Montreal: ICAO, 2023.

BOEING. Statistical Summary of Commercial Jet Airplane Accidents: Worldwide Operations 1959–2022. Seattle: Boeing, 2022.

INTERNATIONAL CIVIL AVIATION ORGANIZATION. Safety Management Manual (SMM). 4. ed. Montreal: ICAO, 2018.

FEDERAL AVIATION ADMINISTRATION. Risk Management Handbook (FAA-H-8083-2). Washington, DC: FAA, 2016.

BOEING.The Boeing Company 2025. Statistical Summary of Commercial Jet Airplane Accidents 1959-2024. Seattle: Boeing, 2025

# Bonus: Veja a taxa de risco para qualquer um dos aeroportos listados no conjunto de dados selecionado

In [89]:
import ipywidgets as widgets
from IPython.display import display, HTML

_dfs_taxa = {2023: df_taxa_aeroporto_2023, 2024: df_taxa_aeroporto_2024}

def _airport_options(year):
    df = _dfs_taxa[year].sort_values('nome_aeroporto')
    return [(f"{row.aeroporto} — {row.nome_aeroporto}", row.aeroporto)
            for row in df.itertuples()]

def _fmt(val):
    return f"{val:,.0f}".replace(",", ".")

dd_ano = widgets.Dropdown(options=[2023, 2024], value=2023, description='Ano:')
dd_aeroporto = widgets.Dropdown(options=_airport_options(2023), description='Aeroporto:')
btn_calcular = widgets.Button(description='Calcular', button_style='primary')
out = widgets.Output()

def _on_ano_change(change):
    dd_aeroporto.options = _airport_options(change['new'])

dd_ano.observe(_on_ano_change, names='value')

def _on_calcular(_):
    out.clear_output(wait=True)
    with out:
        df = _dfs_taxa[dd_ano.value]
        row = df[df['aeroporto'] == dd_aeroporto.value]
        if row.empty:
            print("Aeroporto não encontrado.")
            return
        row = row.iloc[0]

        if row.taxa_incidente > 0:
            txt_inc = f"1 ocorrência a cada <b>{_fmt(1 / row.taxa_incidente)}</b> voos"
        else:
            txt_inc = "Sem registros"

        if row.taxa_grave > 0:
            txt_grave = f"1 ocorrência a cada <b>{_fmt(1 / row.taxa_grave)}</b> voos"
        else:
            txt_grave = "Sem registros"

        display(HTML(
            f"<p style='font-size:15px; margin:4px 0'>"
            f"<span style='color:#27D3F5'>&#9632;</span> "
            f"<b>Incidentes:</b> {txt_inc}</p>"
            f"<p style='font-size:15px; margin:4px 0'>"
            f"<span style='color:#EF553B'>&#9632;</span> "
            f"<b>Acidentes + Incidentes Graves:</b> {txt_grave}</p>"
        ))

btn_calcular.on_click(_on_calcular)

widgets.VBox([widgets.HBox([dd_ano, dd_aeroporto, btn_calcular]), out])


ModuleNotFoundError: No module named 'ipywidgets'